# Clustering Challenge — финальное решение (v4)

**Идея v4.** В v3 кластеризовались только ~20% точек, которые `cluster()` получает в момент
проверки, поэтому выбор k и победителя ансамбля был шумным. В v4 всё «якорится» на публичном
тренировочном наборе: для каждого датасета заранее (на unlabeled `public_data/train.csv`, без меток)
сфитированы параметры нормализации, PCA и вычислено Z-пространство всего публичного трейна
(`Z_tr`). Артефакты встроены прямо в `solution.py` (hex-строки float32, декодируются только
numpy + builtins — никаких дополнительных импортов).

**Что делает `cluster(X, dataset_id)` при вызове:**
1. Точки X переводятся в Z-пространство публичного трейна сохранёнными параметрами
   (fallback: локальный refit, если датасет неизвестен).
2. Ансамбль работает на **объединённом** множестве `Z_pool = [Z_tr; Z_te]` (~5x больше точек):
   окно k вокруг валидированного a-priori (+/-1 по силуэту), KMeans + GaussianMixture +
   SpectralClustering (DBSCAN/HDBSCAN — только подтверждающие кандидаты),
   победитель = silhouette + 0.3*consensus NMI, с проверкой стабильности.
3. Возвращаются метки только для переданных (тестовых) строк.
4. ds16/ds23 (кольцо + центральное пятно): эллиптическая радиальная граница на Z[:, :2].

Детерминированность: `random_state=42` везде. Используются только разрешённые библиотеки:
numpy, scipy, sklearn (+ pandas в этом ноутбуке).

## Данные участника

ФИО: Борцов Михаил Алексеевич
Email: mialbortsov@edu.hse.ru
ВУЗ: ВШЭ

In [ ]:
import numpy as np, pandas as pd
df = None
for path in ('train.csv', 'public_data/train.csv', '../public_data/train.csv'):
    try:
        df = pd.read_csv(path)
        break
    except Exception:
        continue
assert df is not None, 'train.csv not found'
xcols = [c for c in df.columns if c.startswith('x')]
print(df.shape); print(sorted(df.dataset_id.unique()))

## Код решения

Следующая ячейка содержит **полный текст `solution.py`** (включая встроенные артефакты
публичного трейна) — после выполнения все функции доступны в ядре ноутбука.

In [ ]:
"""Clustering solution FINAL — v4 (transductive ensemble anchored on public train).

v3 clustered only the ~20% of points that cluster() receives at scoring time,
so k-selection and winner selection were noisy. v4 anchors everything on the
public 80% train set, embedded below as per-dataset artifacts fitted ONCE on
public_data/train.csv (see notebook, section 'Artifacts'): norm parameters,
PCA transform, the Z-space point cloud Z_tr of the whole public train.
No labels are used anywhere — only unlabeled public data.

At call time:
 1. Test points are transformed into the train's Z-space with saved parameters
    (fallback: refit locally on the given points if the dataset is unknown).
 2. The ensemble runs on the POOLED set Z_pool = [Z_tr; Z_te] (~5x more points):
    k window (+/-1 silhouette around the prior), KMeans stability probe,
    density core mask, candidates KMeans + GaussianMixture + SpectralClustering
    (DBSCAN/HDBSCAN as confirming candidates only), winner = silhouette + 0.3*consensus NMI.
 3. Labels are returned for the test rows only.
 4. ds16/ds23: ellipse ring + dense central blob; an ellipse-aware radial boundary
    is fitted on pooled Z[:, :2] and inside/outside becomes the labels.

Deterministic: random_state=42 everywhere. Usage: labels = cluster(X, dataset_id)
"""
import numpy as np
from scipy.spatial import cKDTree
from sklearn.cluster import DBSCAN, KMeans, SpectralClustering
try:
    from sklearn.cluster import HDBSCAN
except ImportError:  # scikit-learn < 1.3 has no HDBSCAN
    HDBSCAN = None
from sklearn.decomposition import PCA
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             silhouette_score)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

# k validated on public train data (see notebook.ipynb).
K_PRIOR = {
    "ds00": 2, "ds01": 3, "ds02": 2, "ds03": 2, "ds04": 3, "ds05": 3,
    "ds06": 3, "ds07": 3, "ds08": 2, "ds09": 3, "ds10": 3, "ds11": 4,
    "ds12": 2, "ds13": 2, "ds14": 2, "ds15": 4, "ds16": 2, "ds17": 2,
    "ds18": 2, "ds19": 4, "ds20": 4, "ds21": 3, "ds22": 2, "ds23": 2,
}

# Best input normalization per dataset on public train (analysis/norm_research):
# minmax wins 15/24, log-sym 8/24, robust 1/24; 'standard' never wins.
NORM_PRIOR = {
    "ds00": "log-sym", "ds05": "log-sym", "ds06": "log-sym",
    "ds07": "log-sym", "ds10": "log-sym", "ds18": "log-sym",
    "ds21": "log-sym", "ds22": "log-sym", "ds02": "robust",
}

# ds16/ds23 have ring + central blob geometry (see notebook section 3).
RING_DS = ("ds16", "ds23")

# Per-dataset artifacts fitted on public_data/train.csv (unlabeled):
#   d_in, d_pca, k0, norm, nprm (norm params), pmean, pcomp (PCA params),
#   ztr = hex string of raw float32 bytes of Z_tr (whole public train)
ART = {'ds00': {'d_in': 9, 'norm': 'log-sym', 'nprm': [[0.2357389277802339, -0.019149097470070837, 0.023159578971418068, -0.25666900512521573, -0.28220720730899107, -0.023260420788069492, 0.34279160976685163, 0.21676280805102738, -0.01309050171282703], [0.9141977678089253, 0.7865070777162747, 0.7826214131317113, 0.9512544390583931, 0.8390771133674031, 0.8397904079879115, 1.1021415677060515, 0.8464432472520593, 1.0195564022472234]], 'pmean': [-2.7755575615628914e-17, -9.25185853854297e-18, -1.1102230246251566e-17, -1.4802973661668754e-17, -6.29126380620922e-17, 0.0, 4.810966440042345e-17, 0.0, 1.4802973661668754e-17], 'pcomp': [[0.21395779237255869, -0.023164518250613386, -0.003804562109837401, -0.4410291974445732, 0.26861578863337926, 0.48392225967840413, 0.351909101231996, -0.3582122777614792, -0.4479653934194728], [-0.48250847882088427, 0.03241307818082373, 0.03768772255674861, 0.3145487148523053, 0.5248630423666951, 0.1415334631426911, -0.408671526593476, -0.44937398930504946, -0.03621282146264745], [0.008911306528079865, 0.6341779662182218, 0.7708049956200866, -0.022752084944070345, -0.025863784936286327, -0.007260783996714046, 0.03671783186499939, 0.020935054384493727, -0.023932553200050322], [0.04815796002383409, 0.7715495386321715, -0.6328092078452948, -0.006351539862649654, 0.019912362865293187, 0.016371435554378023, -0.011695710092746654, -0.016832333328334437, 0.028629382696284626], [0.6003714553095912, -0.028558958725066697, 0.0617666053312443, -0.1248437840014608, 0.20901061179015765, 0.2631211480432639, -0.3917164109306822, -0.09296396357069728, 0.5868008163630529]], 'd_pca': 5, 'n_tr': 960, 'ztr': '...'}, 'ds01': {'d_in': 6, 'norm': 'minmax', 'nprm': [[-146.82778900814074, -4.502465749800319, -2.8831655718987856, -2.853868171752506, -1.1904358802169566, -0.9755893964094798], [0.003173758227203392, 0.10810239181982151, 0.18278716983884827, 0.1894069869692498, 0.42547719145378404, 0.5387219118712248]], 'pmean': [46327.84898468276, 41.582822539551756, 15.778723917176182, 14.73349886204949, 2.7955651583566485, 1.7957682347843689], 'pcomp': [[0.9999998295985429, 0.0005836941497683234, -2.186417786768355e-06, 9.917183796388535e-06, -5.4371402414094e-07, -7.732940982446095e-07], [-0.0005833506037452661, 0.998484603627648, 0.001629013698017685, 0.05485146662150551, 0.0030425422882576765, -0.0027498874299724547]], 'd_pca': 2, 'n_tr': 936, 'ztr': '1fe11a479ca6b0bfc6982a4555616cc02e182347ceffe540df5218c7fd08bf40d22803c7c0cb05c0085d1e47347212c1b9c617474e7d0fc03b6b27c53f3227c156da9cc47c7b20409cde21c769ea31c14ac61447eb8ed34018b20ec7b8d9a03f5f8713c7082a58bf59d70f478a8f83c011480bc7ac5e2441f28732c5096007c13e8a0cc72aa0b840da3378442ecb1cc03df6df4511f12ac0f25f2247dc97f640865e17c7255ee2c09d66a9455cab06c11b56184745ff88c11016cfc4b721a5c011ac124756e55141eb26eec6976dcbc077d00a476857c03f19e02447bec896c0e39216c735a7064107c60d47e114864088d628c77d2da8bfe9a0aac5f0506d41487a1a4754221ec18e781347fbf4d34064b527c7393947c02a92f2c3a00a43c0f6211dc35126c73f06e53bc372cc20c01ceb16c7e4696340b8c122c7c344e440e5fa0ec7199dffbfe8c39cc504ff16411dad1ec75c3766c16cee20c70ad490408dff2a47f7f436c164e6c7c2fe6c3f40bb751cc7bfcb26419ec008c7f72f233ea1da0647dfa2ecc011f811c63bf3d6bdf5761a47256c32c0bc151f47e1382041c6439c44d51316c1100521471b1f23415b68e9444831d54093f70cc7b0e5513f024839c3a84f33c09b4a10c71a7812c010d80146f724e4c0b60c03c72269b8c0db5c414520e4c840471f034582ae95c0030357c518829a40107e4f44eb26bfbdd1ad1cc78f6ec13f35df11c75cf81b4130b7c74487c1d2be0aac1443c7a0aebaa13f1ec79ec92ac1d51e1d45164eb940c53021c71ee781c0dacd0fc77d49b0bfaf610947dce31bc0f77e0a4746fd433dcba42847cb5c9bbc149d21c71f97e03fdca47345e5958c40513df4469afe63c1cc651c439b76b5bf9ab210c71b9136c16200134734348ec004cb7ac593a5bdbe8b8729c725898e40645617c7c2c55cc03cd81747ee7af4c05e31654491b724c0ae432847cb20d4bf15f398c31b206541c9b123472ff3763fa9e6284736a2ce40aaa23147ac248ec013e1fb46ccec9f3ed9afea43bda938402323a944be1a93402f7702c706140abe97940647474d5f412c4006454a022e4051b50a476cd881c02d5313474e808b3fa11be7c40e941741a82705476b9b46413e1121c74fe20e3fe6e52747bc58b240bc050847b51934c03fd16ec5a86e21c08d001f475a35f73cb99b22c7aebb0241f9f6c9c4428e0341bab710c758eb6d4049d685c592bc89c10e491ac76024494006230cc714bdadc0810f17475c45483b41701bc433df72c18688a1c39a3ae4bdd03c4545103feec0112929c74a641940c02a33c4bfa3ebc0573ff7c6eb9b8340baf01b470f3dd1c04d0921c7278e1dbea5321b4577393bc0799b23c7fa435640484c24c70c0ae44057f9204767f1c540ef981b479992803f744b1ac58eac39c0bcb917c703d93fc08398244761c682c014df89c348ce6bc121d50a478fb1dfc0ce821ac793845c3f08f115c5eea602c1136c08c79ddc68c0c53a1dc7cf7bc6c08eaa0e479cb82ec007dbb8c5b2f49f408e0997455a16564120f63647854d104036722c4797d60140efa1f54211663cc05fbd1947693cd2c0b24c0fc75f5c3fc1362d0bc7fe2c47405e2b14c50e5cfe40eefeeb4647a6fcbf52bf1f47de65663f039710c7f3ad6b41cad414c7d56410c176ab0247cd2ec2bb95db294702f2e740171502c5a92a2640552f3445648a5ec0e3d515458fae0740394cda44c69fb7bf03f22c47abf037c13922e146a396f73f3a6919c7baa2eec05391fac6d7311ec0ff5e6c458f460c402b4200c781940cc1b2eb4443a68bffc0166965c4a984bbbf52fe0947f9aedcc0f1672047a3efbabfadb412c774180fc0a40b1b47e5d3774060100b47c8aa2b41aea722c7d574a7c07bae16c7f4f6b63fbcb60747765b313f715856c53db99b4049fb1147a0c01540de7feec646d37dbf638f16c76bb7f440e894b64401e724c0ea3f20c775bacb40ac4f1ec73402c1c057e411c790585841d62a1fc76a2dfec0e67727474b14b6c0b5ef07456abc77c0a1277d443344cec06f5e14c7759e9ec01c96134500fda6c0b24e0e477bd6b43fbf4e23c636a9803d352d0e47bdd17740fc6be7c41c8f90c0ba955543066142c04002244494613e41ffde03c7c1a1f3c04c36f246a81a00c1438cc44442b942c0de625443d495eac0edf71ac728cd2ebfd83aa243a437b64081b22ac732221140b7e6254700ef39c147f91ec79ce2d5bf564a25c77983a4c001fffa452ee895c01c080b472c23a0c08b1f1d47b954b03f7e6310c7d03c0441c14a3fc28cca6dc041ed98c471e6c93f42c91b47907572c106581cc7ca9de93f9a01104762f3cb4052d43ec34d82dec00bbf2b438ba0aac0bd76fec6726ea7c1a2c06245bd6227c0c4371a47b6d9f2404e3017c77d48274124cf1447741fb240990b86c49742ecbf6048034783e16dc12d8a1ac7d5bdd5bf4d140bc756443f412d601ac71aad80beb8fd764481d49140a3381547f16a89c07192dec4837d22c018c41e47a94819418c37f4c6b66619406e440ec75ad25341bc2912c7bf080140eb2c1bc7c6be65bd88340547ed64cc3fb8c41a470f6e6341931a1847551492400f2623470feaf540b97ea4c54e700b3d9eedd844861e1fc147d8fbc6251d8041cd8822c7940530c126442fc3ee0bc7400ebc1d474e2f0bc083808fc3837415c155f0064757f32f41dd5b1cc779680141467a1bc78be5a53e139b1147e69ff83f196b1a4773332040c7ac144777b578c1f9e325c7a62874401485a9c51588a03e90b282c502c50c40d31ecd446a5dda409dd60fc72aafa340f39403470e93d6407ee7f74445f080c0d5e040442e8e8140a1f40244f68725c059930a45d5ea08401ff018c7644b45c011472447727cbec0343b0fc7636075bf80660bc7913d5b40e6e63744c77ca03ece85c9c431b7ba405f9a084795307e41d7960ec73cbd44411205e844177033c1c9dc92c5b3f8733f5cc30347b00486c1bd0b57450a0bb2c057ff0ac771731441242cdac607cf3cc16004abc585d4bbc076390ac77e628641e0763dc43229c840908a2cc71c6808c0b8b2bac555c4343e4be80e47cfb034bca6561c4729717540bd820cc71f94a53f87ce1fc7ca4ffa405aff6e45edf356c15b3a0c475731553fc95c1447b1d188bfa3041347a18fffc0d5901547943232c1a2f01647c19d823f1a901b47fdb88e4014d31747aba7454176fec0446b55f5be0c2517c765e78e4048e6f7c207a30dc0228fd4c3a636833e9fda1247c1bf0e403c09ffc603b5a7bee7bf31c5795a6cc045d81ec738e48fc150ad24c7903cc83f8ad70547dfc02741dbb51d47399d0cc145ab11c7b5bacfbf550409c7c566bcc0f0c710c79787b2c01227e6c5634489c047182b47209bafc0b5e81547bc33f83fef4a32c7f35e933f6860b94458a67e4019ca0e47e443b441042d53c5c892da408233174728b80cbffed54e4754fc50c1c1ab1cc7525c2e3f448efdc6b0af2cc1cc562547ee58e54020a739c47b828c40509225c7af344840921efe436fb8894028141c470a49dc40d4cc7245ce7116c198070d470ddc324087dd0947b49959c1cd9104472693404043351447a484813e865a0cc7cccc0ac105b91447102761c0303dd245063e12c092a9f8c2337578c0b6dc1bc77aec2ec05af518c7472ec44113160ac7144c7041bc2015471a277340a981ee46ff06e13f2c8e1fc7d85092401a5f054719d1d240ca9401c5e438dbc016819bc49fd60c40a4927145619354c100c942c590e0db4060e50f473abc20c16f83f5444c80a5c1a19d1cc7f0a17fc11fbb1d47b30992400834804581551340730c1047a57253bf49e8144730770dc0881c3147fdb31fc18de0e74349ce8b40dce21bc7c2eb21418ea753c513b01d41c1df19478ad97cc06a9f16473bcf3ec1d55c26c75ab234c0534c114554187abf1663034795564f40dc0a26478b6b3741660719c7545042c0992e03451bfb1d4046638dc5e1b7fac070901dc714e28f40bbc206c7a91c234167662747ffe743c0d7058e45826ac040689f9245b2895a40447f1bc68b9fdabe11610d4780c9edc0a1a105c7d0a40b41f9127045340b27c074b9efc60b6b4c405c4af2c6d0d2964074e317c7ba49b83eee85bf437f26cb40f6b687c488d0c2c160772d4758a07b406508f444bf19ebbff65d10c722b4d83e6a0f20c7d2e915c158ac10c79d1235c1477a0c477407ae400e2c15471ba432c1e0194c44714c0140c9881947e54919c1e56725c75054af3fe47b1a4785c5e9c0a2c012c773b6c73ee1820847430b1a407dbd0bc7f77380bf96acb945f94d37c0ba0f1ec7a4f41a41dd9107c5ac5e7ac0d68faac5e14966c04af6a8c5e08f9c41bce1c8c4a9b7cdc046590a47d4bea8404b1a0a4701001d4124a68a458596213fe8a174c5169888c0642817c7d07d04c0ad57f6c63225a7c0477e034709e01d40c78b6545a9ea9b4009f3d2c434a42ec035b30e47a80747c1e961fcc6d0d32f40c4d723c7ef9b8140acb060c583a0c640975d6dc564f3e9bf7601e045f643763f2c9e5545201541c012370247e4ff0dbfc70b61c52b887cc09488244468a2cbc06e8a1d47ac0f7dc07d801ec7083524c0276912c78324aac0683f1ec7b16591c05f4b01c7aa7c33bea65c1ac76e9c77c0618771c5314da1405bbfe344feae6440befc0cc70b6d75c0c70a77c51e17bc4082701f47bcab0840c87421c76632773f419911478de191bf354e0a474ddb5c40db4f2047440ca73ea4da02471db97f409bd515c786224b41839908c7a7e295c0def027474d4c40410dd413c7a313c6c0801f92c450b076c07d9e2147839c8d40fcb311c75c3572c14b1f1847f6792d40d3e702470af169c16d6e0e472da92fc0c77b0cc7d4387740ad6b12c72d434dbf42ed13478bc8b540ad2b0bc78f1d53c0645f29c5b97eb4bf666f1047e8245c418a32144738000341e1db1e47dcde7741a54d07473d7cf0bf86bd0ac7a652b33f2a8b14c7fa20f640bd6c1bc7ea7d78c0c3b271447d932841a88ffa44002dd8408d8be344b08b02c02e9709c70d655ac036a8154788ec10c11c808a459629d6c074e71647173dacbf687ff1c28959a63d854248456290a540c7c20d47ed4728c09e240dc724bfb4c0b6fa09c71ffcc9c08c68e9443b877c3fe5b1074797c18240a4402a47ea7374c052547ec577fca240994a2647244ab0c08d7a1047ff9d5c41f11305478918a93ebb9925c72a5652bc47e412c7f07be9407db51b47aa50adc0b4bed5c3ded096c0b8fe28c56a069cbf1ec19ac5489225c00abc1fc767dd16419beb1cc73c8507bfbe759f45c9c0ec401acc05c7289227c1efa605c7c0ce8ebdc6a442c3f62d853eb8c197c41c81c43e19a898c4c50306409b311f471ba29bc0db2a0647ba4e38bfc6724a455cb939418b8e09c74bf0903fb9566dc4133311c0d6e920c788ab88c0e2230e47be0365c086d209c7c8b214419acf63c432142e413ed903470d13ac3fae100cc723aad73eddcf1cc7eb367ac0109277c59651e1c01c57d3c59f0913c1df5024c762c793c0a10223470d124541398b16c7df9a95c0e4900c47a9fbb041d196274454f1f1c0e7fdb64494a38040c8bfadc518262841c6911b4785869e406ffa2fc768f6a7401be22047c56733c10bac23c71e2481c0327455c5b5b503c078ad5844f9de17c0435f0dc785317fc0390413478d9e1f41a5f40e470d4b3941f49ec4c5b410d640c1b7ecc4f83c2c4091ddf74265b1933e4d5916c77d0529bf6271e4457b3712408ee4d644551016c1c6261fc7b3ba92402466c2c438bae6c008c419c7d0304dc0934c1745cebee640db7c2ec66fef33c0b2c638451ea70d4044580ac7c4dea1c038e51147a0e6dabfb9d40647c9bc87bf5803ff4684f70e3f15b718450f150ac0d5e50ec734300c41d889124796506cc1ad5e0d47be30383eda214d44c3665d408c4920c56b5ebb4068152bc79d112dc183db79c46e355d40cd7d1247093de1404de1f245c18373c0beca02c74bf196c1ef821dc795d1adc09de7014728daebc0a261c944b5b7a240fe653ac3293315c0290f2ec7452d404178d50ac7d41344c02a27aac5f32b093ff2991cc57d7ae540156914c694b669c081ddfe46461b94c0a9151dc76a63f1c05e6014c701724640ca3f184707652b4164e92447225664c06c8cee46fd6a1dbfda9514475b057840488b024725f6fac0f1cc42c49062c6c0a4d927c7b3c78bc06e1a21470cd2cb402e7f1a47291ecac0dc92f846f21505c0e380ff460cc784c1f01d09c7b9c7a43e76981b47546727c01fdc10c7baab84407bb028c78b68213f2c1c0c470a17bfbfa721b1c57a6a2f411a1910468dc53040c49d2dc7c6eb4cbf7ec40bc7549335c170cf66456783553fe66b11c75536844030aba8c5ee6b2d4018670045159fde40fa840d47c8e2a1c0c4e2c8c48127f63e7e610ec743f39f4098df1447c68291bf15e81d47b4fb584045dd13475a3a42c0847ba144c7f3944073c920c7b1958240e82e0647f78400c0f79713479a49973f7f640cc73dad1d4177f0bf447ac2dd40afdd17c7f805474110ff1845f22154c14a4642c5b9db2d4163f7e0448d099ec014d570c572f0f13fa4ba1447b31d4b411aff8fc5532106bc3012214746ff05c1350914c72913babf637d1047b664dac0feef1e47e5d7d5bfc3d69fc592fb01c194a029c7e57857c0ac7f2dc5f7e2ec3e7c302447544427c0502416c7a04fd13e306914c76796a83f280e15c7614bee404fee0cc7acbec4bf082526c7e133d640e4f929c7d5228540d827b145f0269cc061d41147c6cd70406bc4e6c6f6c6f73f21481cc55472b341da9a2047cd1f0441a0387142f327f64059a2e845cf5f61bfc6dd2847a77e1e4199a418470762da40405fbdc440b8df3f17fb24470d19b6409b5415c7e20c09c0c95cae452243babf7b8819c708384f406e3a0c47cf9a1a4116030cc75e5d383f09fd1647c8be164005330f4798c00cbfeb9413477a77433f23edffc6b954f8402d3c4945853bc63fabe245c5235694c07f831944660ae04018bb1c474ea4afbf86bdfac493439440f3bc2047086010c0910889c5a25885bf78ea19c7198d82bff56214c7ca88cd40a07104475c700d41f4dd1f477fb6bdc09361cfc640e686415c0710473cff6bc1f2872dc775e3dc4025700fc716d0afc0e10b5945d9f8a0c0af2a0ec785062141aee11047c8ffcac0caa315c433f9af3fa9ba194591eb7540d84b11c7b9b5854117bd0e4712ec7cc0b60206c73fca03c070ee1ac443b981c0ba70a544d24c6d41881805c7c7433741643215c78ad648c1194d11c70a3505c177845dc552efa5c0c012054791e6e23f22451dc7e4b283407a443dc5150c7cc07cb10347b137a73fecf2a4c453aadf405c7f0d47e33521408067cc43454c40c071d8cb45da3bafc026a515c759d81241ed392047651f4c412c9e074780276dc058231747e227aa3f80d562c5b284f44052fec14439fc0dc1f30522c7033e9dbf470321c71ca2d3beb42adf44d2cdd0beb7a128c7b6d02cc14d751cc77d7fbe40afceb1c5be540e40ece00547349e0fc0567e0ac758cb793f692f04c749caf7c001255fc5927b8041ec280bc71b9a03c1272728c53af22c3f782616c7e3a8f23f025d1dc798a57a3f2f700ac749365940bbbf27c7eb98f140ff6a35c2dbdc9b3f4dc61b47e4a9813d572c124796102cc0b87b16c7f1b8c240dd9b9844979764c18ab4194704998b3f435210c4c10e34c00e73014559de1541766e08c741d91a41426bdb459d01a84002f31fc56715344109f10ec7ef7040c1f17d144728164741f9225245503ecf40c2fe154708cc433f2fe50447c14ac7bf73621cc7654073c0037461c3bad583413a3c0a47ed1d10c06fcd7cc59b0d36404f2319c75c4cb5bf90c49844967f4ec00a5c10c7802825c138707cc5c2b931c1514699c491691c3efc91c2452fc752c14d0903c78b972f3f4a3a06c71521b2c0441bd5c5d7b2dfbf3e3801c79c8a233fc534014700fdde3f01010dc792e8b3c0c35f75c4aec7ae41cfd20d47d68db6c1d73810c7d8931ec171bf0d47cee01c4065af0d4700530d41cb9417c77624173f6b7fdbc449d90bc00f89d8c3e6eac2402f4c15c40159d340447b2d456ef1fbbf158d18c706c8c7c0d2530447911953c145bf0b4733f7a2c02d64d2454061a53f386b2bc7f6e10bc0986d844598706cc091ec04c7185342bf808f22c7a66716c0eba902c57fd11dc17e1b2bc7abcef0c01034dbc680988fc0321a08477c3e8fc1762214474a9cddc00a411b473f1e8f3f931a0fc735d279c000bdb4c44f042b40b9760747cf59d3c0b6ea26c745761c40b7691447c11085c067801bc7264c12411b366a458925b3bfe79efbc62d5bb3bfdc5630c7a7685cc071fb1547f8dbecc034b90947feade53e75c30c4782ea1e41352021478bc1e83edf1402c7d1dfadc083fc0c473586014147d2e743424b833f02eb124766628b400f1d0747925dc6404e96094724b6194060723dc505bbecc0dbbe73455bc180414f3e03c7a80929c1d3581dc42c6dfc40c1a606c72dc64ec05a9110c7034af6c0b42529478aaaf2c09d2f084796871a41f9aff4c6213b56404b4d0b475a456ac1e22518c797a7893e9eef27c5b89700bf6a173247409e89406f2e8c45e3946f40be9805c776a12fbeb6301547cf31803fea5d18c7f2b582bfeedb8c4451436cc1200e30c3f0be5d3f2eba3747a06427c0ecf322c78580553fb0ee13c7d6d7b9c02dda924513fc2dc1ea93c3c57c468a40ef9614c711152bc07ef280453f870dc1360911c7b36a37c03b6a0f471d4c0bc05bfd0dc796cea7400eb11bc7b77230c1d63dd645fd23304181a80dc700a4f0bf2123094742bce840b6d9134729b33cc081140cc7885d81405e9509c72a6f4641e9983545699fb64029341947cc715bc0d4115045d9cac7c0d67ac14403c78f3ffd8e04c7877040bd35c60947637fd9bf46bdc4455f74d7c0ec159c447fe983bf1dc51ec7e0b0e7c0679921c710de2e400471204705d80c402a2823c784e317c1cf28f946f2f51040dbf734c78d645d40641a6bc3a2515bc1a87e004776d9214085b21fc7d830094192f30447e13f90bf26b12e476ac1f43e13ad1747e6959e406dc711479f3f36c152da06473cd9a2c0247e0e47a7f0b740cc2e2047f99e023e3f4dae42899cab402802ac45b830cfc0fc952fc557e2dc40aed10fc547c7b13fadb516c712be70408373fdc629c54340841783c424e4813eefde2547bf3357c0f779124795dc243f97b51e47028e8ec055570ec7ec0c9f3f3c195bc45419aabf94371b472863124101c01dc53d33da3fa93812c71c58a640d8f51f477a3842c0d4d709c7eade5241b8c81bc7e1267140ac5217c79307984061e50d47aee3044144a816c5844bc9c09a62d344a954c83eff550a47ea8d0a418a6db4c31d9882c096291145deb6afc0d9b062c5d09051c0407a0547fadba43fc4280d4774ce6dc04eb11d47a810e73fcfe708c7c37edcc0e3ba02c72c2d1dc13c7d8b45d7d7bc407ca010c56c5438c03f192e474ee28cc0a7253d45ad16f6bec6e511c7c3ef21c0dae0084533b63640bbb526c7d03e31412e6d1d479052e0c082941bc7658adfc01a7929c72d857f3fcbf022c73bdb0c3ffd6301c7841740c19e0b10473ba73a417c910c475a54dfc0029d3245a83b963f54bc0847cebeac3f07d31d47d9aa693e3a871d475d02a6bfa66905c7d20d0f4199a85b456cce704059e818c74ae5ac400bbd12c707b0a23f5e84a045e4f9533f87e72547d45d7e3f72a823c7b6f5d9c05108c7c438716340cb422a471acfd2c0dfd908c7fc09b83fcf3ab6c32534c7401f3e27473c9ec5c0518242c545cece4078ac22c75583c9c088713dc5cb9e21417ea11a479c6455c010131247465b35409ae363c5cea945404c0800445207c6c036be9c4449b78fc1913c0f4776064941fc8a094723fd0341a4ea1247a4a4f4404ee91d4771105940d7d32047181c8dc0d3d01fc7205d65c088110ac73d1e6e408a528245e01a0a41011c1847b7f5f13fed9a1b47c515e1402afe1647be8626401ef70f4719f7dbc0360103c70ff54f3dea8214c7fa7dfcc03aa2f64377f2543e06018a459d51dec0dba9074726b23c40aa8902c6c4adf840ff9006c7cdd63ec1282b2c47bbfbcd3ff1bb7c44ec14543fd4007dc4721d584094df0bc7fe317a41cfcb1cc7a765b5be9c8c2347b0de6f3e3fd01cc7ba4920c1eb4f20c74b63cac0e5ae8e44b230c0c0ccad09472f86c1c0d2901847cbd1b7c0c63305c798aaeac0d88e12c7c798a4406d7c0cc74c3ab23fb3d806474671a5c0b97c0f472c335c3f3e0115470c580440b3361cc5610f8840b88311c73383854097611247fa44f5bf805d1ac7c424dcc0fbe717c7d5f67ec0011728c741ebf63f52c90f47f68040c09bfc1bc78f8346c038ac264795af3641'}, 'ds02': {'d_in': 10, 'norm': 'robust', 'nprm': [[-0.0979974472710166, -0.6589315790940987, -0.00047389090700904993, -1.0964115786989956, 0.37410662596700284, 1.0692010214763854, 0.0224825796087239, 0.20044181704779315, 0.40344454594188106, -1.047141643925473], [2.486973489519067, 2.57020478560384, 2.0332678437931215, 2.811525322605766, 2.5644784071376177, 2.871717277150323, 2.016891093251873, 2.28289340041201, 2.123690378571906, 2.3460489979358]], 'pmean': [0.23260779265097756, -0.1361320521669722, 8.002855875567801e-05, -0.08912074733446237, -0.18951687387631633, 0.10986894307784724, -0.013431172435571617, -0.004560670929560244, 0.036455827014832555, 0.03439384335292578], 'pcomp': [[-0.5398674738542752, -0.432855905323426, 0.053311773601562966, -0.31510211811438327, 0.5426499270060227, 0.10543532544173084, 0.01899358763771955, -0.220912900949682, 0.23072988875466566, -0.10517740719082529], [-0.2946359472341023, 0.08204282647523195, -0.06731524573090668, 0.38265215567472066, 0.15445026322508443, -0.18884453486058986, -0.04132233126567792, -0.4350993890725486, -0.4293812811395545, 0.5662156642910063], [0.3108858972450849, -0.3946043858334608, -0.04470360361084553, -0.19609309316265852, -0.2165755666886972, 0.6409738483582132, -0.013007264638642687, -0.34850605421534364, -0.34666207436484464, 0.08737116965274826], [0.005115044277924561, 0.005995221629016084, 0.09847287953952236, 0.022653198850642525, -0.008298348866596146, -0.0010356381391152325, 0.993527663069284, -0.019273060261736853, -0.03438610577060043, 0.03173028002685349], [0.022820936322809047, 0.011075536264036586, 0.9904183986320604, 0.03055954471076329, -0.02743558543669269, 0.010109952081438908, -0.10320003882825224, -0.03165482260798545, -0.0542122967965171, 0.0452534620575921]], 'd_pca': 5, 'n_tr': 1010, 'ztr': '...'}, 'ds03': {'d_in': 9, 'norm': 'minmax', 'nprm': [[-364.6521489383634, -294.7200133194208, -363.4734601056708, -274.8523731785814, -33.8242778918671, -246.8965866857081, -156.41326411123927, -268.9149388137938, -535.0134502599777], [0.0015317798081497076, 0.0015008516056016139, 0.0015341939864808986, 0.0016094479892576768, 0.012969001410715546, 0.0017894451439767086, 0.003562411396007507, 0.0016449389635694851, 0.0009855525315717875]], 'pmean': [245694.44176267768, 187976.4340713039, 244898.05893746708, 163481.09312566507, 2511.3173252695583, 132291.16301762982, 45393.496356477735, 156496.95048988945, 531281.4792735049], 'pcomp': [[-0.43732477815108733, 0.4558849024994288, -0.43632141020499154, 0.3965112647928163, 0.006100159855389223, 0.32044212892641405, -0.08091056641659286, 0.3795370563976825, -0.0017807580668069444], [-0.0008422870257871782, 0.00078685113339706, -0.0007291510923003733, 0.0006896224373933512, 1.717892211762785e-05, 0.0005237624753771203, -0.0001386185840203706, 0.0007455011945848398, 0.9999984072827569]], 'd_pca': 2, 'n_tr': 960, 'ztr': '3b66dd48b01395c7934f074955df7bc816aa6dc83ef3d647b323e5482b8381c86a1297c8369e76489b1731c8f47c0b4573688bc804fd2c485672df48f46063c7aa449348ecc7414888dcc74835ca72c6190d83c8239900482d0f88c8e72ab547368cbd483cd40bc73f1c07c80398e3c5141f7bc81031204869632cc8eb8af0c719ce3fc8049c90443b002cc83d999f46823f93c7077569c82f3367c8faef0bc8e3ed3fc809ea16c8dc8397c856814448ddff97c88e77184719d6d5489092e9c70f0551c8ab4098484dc3c94862e6b4c707028ac84339a4482bbe5ac813d8a3c6ba482dc81b453d48c572a0c87b11a5473dac25c885d44447a8cbe948c89905c7ffefbdc540753745875a5cc8a01714c798cfaec8d004afc7036ff24813a3abc64be956c8381e05c8995525c8a0a1e7474b2dbdc769c15d45cf1a5bc81f4dad479593d2480d213147a0c6c448dfe2cac6b9519f484485ab4714e892480a6b70c878a564c8562a804813bcc4481bcfa5464a1f0149f12fa1c7fb2212c817adb047170d81c8854a2fc75567d548782c41c79d0ec34854b5284845e94fc8a4e12b483468d548c8d0f8c7da44ba48814bb7c7feb67cc8b7e07447f768bf48a3943cc7793b94481b5bea4725455ec8d4491dc647cc8fc8ac164c46be93a64854397bc804edf448d5a54748f27c48c8b8101448685ea248a0a6e447a58e38c8820dbfc7e776e848858abbc7a49f47c8a51ec5c7866ad248988a994715f5df489c6f0e4812fb74c860a389471b7753c878a1e6472520ba48b5db90c8046d69c8bbe97bc844f48ac8cf681cc8e960da48791c5346117c9d481a272fc87b50e348ad1a8fc70c29d4480361c647dba914c8e938af471dc464c88e77bcc8db19b548cd7ea0c88e315cc813b25dc8495aaf48b2892ac6c4192dc87f0161467e9303c8978e3c48399386c81c229347872973c86feebe475e2acc487d3d5948846cf9c7870264c731bfa448e469c4c7e46287c8e9bbaa48678c7ec8977168c82a778bc8ff569f4739cf53c854d2ae484ac8c048cc600848f1b5da48b08bc2c760f745c8806aa8477d5ec24879d4b9c7610083c8570a11487cdc05c85c83d4464a7178c8b31451c60223ea48d1bb29476221f048268318473cabf84817d5a1c87b62f0c7f9a27ac82e3531c87a3321c8b27e66c8bd97c6c79d2cea48c43896460d355ec8c5b977489fb6d348be7d6ec84d5c73c831a04c48c2b7c248d44b83487ae947c80892a047be9563c8a7802ac718f728c8f94b1dc821f58ac870e3f6c7f51b6cc8997420c8779b53c879708dc75ddf71c86bbf76c81f8ebc48d1ec3bc602675ac868811245e6d957c8b2100a480eefd7489836dd470bc430c8c432ae462adbdd48af1036488dd7c2481437ee472388db489db333485f2b5cc81c5c1848b2f09fc8adff55480ffe49c83124d0c6c5ff93c8106a83c8d454fe489befb7c75d4ce748b5a20fc7823788c8da2cb247feae5dc8968dda47463f29c86883ed456e6f0cc8f66842487bbe0049602bd648a3814ac7323d30c7b92023c8dd662548fcae84c897756d488831cbc71f73f8c78c6034c87f7c88c740ba77c833a7aac766d8adc815edf645c9c570c7379434c87c97e848fa6d82c8a365c04837ee11c86add59c80e72844744dd8bc8b7e5984726baec484cf48ac848dff248db5dc8c8b4e846c86a2616c8c13dce487262e6c6fe9dda489a8a7d45002a52c86ab674c8c4556bc8c56cddc7d10322c8836044468df955c8047b11c8fc2acfc7b76fbfc7d20b00497854bac75747dd48c7b73dc6146902c85a280747c50b6dc8265963c5fd72bd4857e3d2c7fb5b92c848d3774570c3e8489711e7454d7d64c8a448d547b34707c8442126c8b4622dc809caf54584e937c8a4fe06c8d361b84839985048adf3c7486c55ce478afaa648203a6c4774b8ccc72ac346485e1cc8481ecf0e487439ed488c129ec64e9986c87f0d60c787febe48c4ecfbc76bfe48c87d97bd4701a7cb48faa4fc4710feb348e0ee24c8977d7bc84a8dbc48bc58b84826c3d347534f8ac837e940c860d8be48cd3c23c8d7f2d448770d45462b8ba948cd8c70c7a1488ac8a42bf147c5fc4ec88d14f04816b621c88b0ba6c718b3b648cd147148439bcf48532b5348454023c8f277ee46edf896c831ee99c4adf172c89d2b3a48195626c81a6f3848ac180ec8dea9c04763c5a7484f511fc8e4c8f2c7a64f97c7362a74c8ab1488c7994675c8c1080ac7e591ab48208b3dc7bde073c8b39ed9c77b71c94805311f47100a8fc838918b48d9a664c86153a5470e006ec8dadf13c6615628c850d7ff464ed445c84bef25c6f50de74813ffb9c7f2e1c548f416dcc76c7789c8f0cda4c8892354c8950418487da7ba48b23620c74bcb71c86ba49c4843b7e3489f204147c70d8cc897acad4679402bc8015880484318c7c753c750c7d8530ec8ac116e47c61b57c8571f89c7b7e23fc8b31335c77254b44883f8ed45c6134ec8148e1fc8c37a13c8b0a169c8bda1d348cfc6fbc5e03ccc486bba26c8ae8879c8c7c1cb483988b948200394470c89bf48ca17cbc7ba560449b1a6afc82826a748099f38484897d448a49f05c8e6bf78c89486ddc6b7333ec8549ef747d8fbda483ad2bac77d8999483e914ec807402cc886311dc8172036c802db0ec84399c148fc46d04796a430c8294f9fc8dcd8e0488bba64c76044af4889f75cc70a9ca0c8b4b437c8fefcd048bc67e046966e50c8331e30c7008be7480b1240c7320bed48f7bfb5c7915e11c8a20986c768ddb948911f45c8711c59c8239f6cc8e24f62c8c7655048a14493c8bd2192c8bd9d78c8291f8047a868d5481d43a04784a24fc87d55ecc7944895c81c62f4c62eb55fc8b6bb86c83da151c8f1ec3ec67ce422c820f864473c9c00c895a1a8c7eb0b6bc8634b7ec74fb243c80d6c6147fcd3e148f7bfd647ff3583c8eebe6ac8e9fd52c8f59663c604d4b0c8c675c4471a10da481af256c7c2e6ba481d168248762046c8408b9ec8b68482c89ea4f7473c5073c8bf570ec7763943c86729be4817d846c819cd714821001ac8aa632ac8876761c8237fa9c53cb20ac80ec4f1c8c01f82c85fb398c78a5689c83c485548ea9377c8819d8e47d32dc048bc4ac5c5bf85e748a25c1ac82e8e00c867fe8fc8b37624c8d6aa18c7782872c8cfb0bec736ac03c71fd58948b86fe4484a63a847afb05dc8d7977946a71ec648fb12bb461c7b08c8322a13c8df23d9487adf04c8cdd276c8431a18c7e2e93bc83c15f0c75e4ad94893934c48434894c8f2d601c90c04c748a7c8bec71eac8fc8b366054883c7cc482fcf2a47cfa419c8847d77c8ee1827c8cef7e4c6781e8ac758f31ec818be10c87d34b9c7357142c8ac8ffec719cf5dc81f5b68c8b09e16c8820db2c67510d848440750c72a9173c8cce8aac7938094c8d050d2c762cbcb4886706b487c1d19c86c9c0ac7bef055c8d6311ac8f2a100c84a6806c841389e48e84ba2c67843ad48922538c694b9ee48c1fa0cc83f206fc8f807b0c7d99eac48a968d4479d8813c8787b3ec7b88e6cc894d4afc7470105c80e0f8e48d080c1c76a39b44896e273c8c8df11c848e4ccc7444c4fc6f7ba1fc8c044b4c7cf6ddd488843dd46eb256fc8408f26c76c7cb5480ead3bc756765dc8e5d493486e4c2cc82b342fc815b485c827ee04480454a7485800d8c6a887e04856278b4551c76fc8bf520a48760393c8be2a13475692f148785f7ec7e26a5fc847877dc8621c2bc80607904802d68cc858d3f6c776f1a24828a328c7fa3f20c84c3fa7c7ebdbb2c7220ea6c791840fc86d73ccc66a34b548fdf1ae478e0c99c8c8a8da474a08b8482e9606481ba826c82d4bc7c8caf85bc84be18fc71f1521c8d0893046b2d670c8016e1ac8ec8671c843680a48838e44c8e57e4348a6dc41c8e98712c8094742c802070ec8d0125bc8bb38aa4778b1ad48403ff147bb62d1489d1f8b475e131dc8f43868c7b35866c8fb0294c85eba2dc8c862b9431fc387c8e45654c24b0ef6c77ad11148092743c8215ecac679928cc8915a33c84bc231c8f5b79747233047c89b910448818385c8afce74c7485f32c81ce121c87fa839c8ae58b3c7ceaf45c876702f48879b54c89c6d83c7d1a58bc81f40f44660f638c8bbe1b4c7ed4199c87dc48947478d0749418d09c703a9bf48862d24c70fabc8486bb0df4680e53dc8bdfe84c778ac7fc8a9fb1d485a3d85c80ff8f14739f40fc8394d0dc8b5a887c886c87547b22048c83ae01748645a85c855c2d9c8be6fe848fd6372461e1d51c86cbebf48584df248746f0a46b35461c8f476c547ec242ec8d9b54c48aa1333c8b67cc8c7900e58c888c7b247167d94c89efdf0c652cbcc4807a30bc898041cc8c085adc7667ab84835c976c72155a448c00da0c8bd858fc8b11f22c8b0aa60c8f40b8447933781c8e1be54c8756841c81fea40481ee1ab48cc8476c886a90bc8ea285ec7be4875c8bb68cd476203ef4835223ac8bb7d4bc8656989c79a00c148893824c8ca9d24c8476f6447a65be648f6740d48ff4867c8ec347cc76c4545c856d855c7b41b76c89a9b9ec55d9676c815a08248a07676c8c15d50c8bc2881c861d8ae4659e572c84e100048639c40c8a2ec59c87994a448a8b91fc8e56c46c86d039c486578f4c7dc531dc8b3737ec8a4a37cc5069692c8e6cc4ec80da924c8dba682c78c0059c8c8efa0c76b2ff148ff570cc7d45630c8a49c8b48c57e71c822f804c8782634c8977a0fc72bcc94c80158cdc79a6d45c8727de847d46e5ac8fe3a10c8c7f4e148d478fd47f94d72c894ad9cc8f95363c88d9a70476e4e71c82ba195c7ef698ec8e5730d47c661e0480ed30dc839a76bc83367dc457fd137c85103d2c641d365c8628617c880866fc808dad046b3e551c83c7991c70896e0488dbfa5470e05ad4810b38dc7446fce489857b5c7d19cfe485b3bf1c69be803499dc8f4476d4e7fc8ef458e47dc6360c8d79ec845cb169ec72e233648099335c85ee258488657dcc7c1c69847d24777c878ff91c8e7ea37c82bb3d348e8be67c8c0b2e2c797f0d948840d9f4857b49f48e006cec5c30cf748e02280c5736266c8be0c19c7ab38f048e6fc05c8e6e533c8d04567c8a601bf4891448cc74bf35fc8042ff1c64e47e24863c95b487edcf7486547a6477699e148cfb38748370e44c8991a0348f4487fc82d4e39c7fcb193c8b5c634c8e60a52c848fbfec71868d548852dc8474e7f6bc86cd86cc791504fc824c2b7482138b94881bd2447359420c81610d347cb664dc81a35b945764ee448fd9af1c78514a648589adec551c95dc8fb820b48f3c096c883d9c848dc6573c854bca1c64446af485dd201475ea8f24817c05b486d532cc8fed6d6c77ef771c813d2534816a2b44883cfb747e9e5edc74f66cdc6503a6ac805df3347159389c85cb0844784ba8dc8f7482ec896a665c880611fc857ec6ac83b41d347ed0304c8bfc8d6c735cd8bc8dd7553c8a588d0484f8e604667c74ac86a0a1745daa468c8226246c8c5f469c899e408c7146256c8997573c8457460c898d787c77ed48cc83acb82c7b3d452c8fa727c46443735c86b1607c8148fdb4811e7e2c7017935c8f5e0cb4766e74cc843cb4448600ef448491410c7ee1ce44812038f488cf2cc4848e803c812b2d748e5a97e47efcf56c8dabc57c831e2e648e04df747b7976dc8a7283348c9a041c818f380c8b50c47c8127087c7938f79c80a6dbc47913a58c8050775c8bd7b4ac8232aa8c61fa960c7b479b2c642d42ec8866260487c4ffc481f5fdac8ab2c29c8b87bee46ad5962c89bffd647922881c88c7c4048a77d55c838bd7a482165a948e81a16c6cab2c4483e21aac8c18b2cc80d1e92c7fd2235c826ce47c8d9e3e1483e28574682a9eb48d5b29dc69dea4ac823955248ac957bc8e8cf9248fa48ac4803c614474fd668c83f1ff246e8c99ac8bf155fc805c6b548cc2b48c60007fd484cd68ec731fc27c868654c48033cc648cfd444485dce78c8385ef547de99c948320ebcc7d1a4d748273e9548b6560ec8d342a1c7ce3deac74f2385c64c081fc857ae8148320318c8c6131bc88d9c34c8fecb86c8ab4028c86c84a4c71c05e34893b30dc7b85982c870013448a196ab48014c19489da163c8b255d947173927c8d2f74648c3230949bdfcffc76ff774c85a45c1c7fc9880c800378fc716292cc8cf1377482eaf34c8b3ec6348143e01c83f911cc79cf4e548434af647944699c8aa1fd64729f771c825eba448d1781bc8c6904fc7556c69c8626bd0c7cd168fc8363dee47374f4cc8b0eee0c759f2f04892c4104895cc73c89f3c314896498cc8a59976c76def83c875acd0c6bbd474c828889547c86d6cc86c8d9cc6ed7aa6485cdd44c5e75ae1486cdef8c74e8777c82dc0f847b0fb5dc866aa82c81cf10bc8073e28c8e1b812c87ecbce46ec46ba48bd6e9ac8667036c82fa58cc727c827c86ebdadc77be59bc863074447ad492ec8547071488231abc79972bd473204c7c794ada148943713c8aebe53474e1eeb48f38d04c875ec19c8e4a32cc5dd240fc8aec31c484fb9bb48b9a99f467af222c89f20ff47bcfa43c880b2fa47306da8486f7e5047bd0ebe483317324831efbd481fe48148d9e3da489ad49345f1d6f348fe9810c7567edc481c058dc6ffbe4cc867afe6c7249bb848341bafc7179d2fc81a6efcc7beed52c8411c0148056007c8baff83c62e4dc54860f4d1c64fe469c8899ba8c7a6c4d8486b963c47ae3e4cc850795947435ab448bdafa3c8add2fa487c4237c8b19a11c8a9101dc8d2b2b348bf8e27c62d0baf480cc0b847b19ed748f4176048a502e6482828954738d3ad489e3191c7659728c8abaa8748513520c84fea12c88970d548e4b48a48254324c828ddc8c7204a97c814d00b47c83681c8c53ce7c6300b1ac80d791347b5cb36c749298b47f01268c8fbaa7c469e9043c8af0c3cc82d90b148cb7606468584b048d9e2b247c1d534c8db4426c846d979c88a2da0c71ecc32c8d22d1fc786efda48a32f984876ddc8482eb46ac8d9de28c8c2eb15c8242889c8843c0348027033c8f042a44755e809c850169ec7b6cb64c893563ec78abec24844294ac75655d8c78c1f184885783cc898333248f5dbab48f19719c88a3381c80f32d9c75eb778c8de209bc7f962a84827ded1476bbeed4811ca2cc8ebcfde485a0381489a9b42c89ca2f7c6887e86c89e3e9bc8a55962c8a3f008c8d6fcc14888fec247dda04cc89afc33c8e98e41c8f811a447f67bce484f0ca4c7b3fad5c756c331c81702c84800539bc86bf09f48f89445c79fdd30c82601e5c71f868dc866e4ba48ce4030c89ca83e471368e748912260461de204c8b4d1b746d8d0e448234312c8db0cc9488072b3c6ada8c648de786dc8f195e9482d7d88c7b4ded24819bf2a48446c0cc8bd0b9fc869edc048070cf746b40b80c8094c2b48323ca3c8f5bb0c4820063bc863345348e93591c84c5d5dc8c613a848c36f6dc8f99a2bc84d1dc9c637972fc8249f8248ccc2c948b60d8fc4b33c2fc884ae1fc89ee583c806cc53c776d446c83c370dc6b40910c8bcbdc1c7882067c8f29b3f46127420c8a84ce4c752a437c807efbc467cabee484f711dc8a5f320c860dd94c81cb7a3c89d085ac8dbbf35c85753f845348b27c8d89908c5f0bbd5c714201f48f43cc44851dbc245e503be4842f118c8d481e7489883a0481c5436c86f9b8348568d57c8b9d12fc7e62a0bc8537f30c81957d048d74d70474ae69848542297c80d4b61c8f8fc2948c738d6482c4d44489e1fc24834eb91487a121fc8ef86ea4602cb6ac823f425479a4f4fc8106fac47ff771bc834308d482460d24890c69248f23ccc4803bb92483f3315c8deb7c6452d1f85c839427d47d7abd348b2a64bc8bd3e83c844fd0fc7f79247c871ce02c8156c5dc8fb4956c8cb0910c8cfe51cc832ea3fc8a5fc51c85f5fcc48db138a470c2c2ec831a0a648d2a6c5488041264866eed24881c16a47c359b048780dabc745a32dc8e2874fc8be5166c87bc5d9481b4287c878e2e147d7732ec86edd02481cd818c8de6131481c1ed4c7afac4bc7ceb03cc8979c3ec8ca917cc805bbbfc7d015d6c7591100c73f3a94c8e7983647599f59c80864ae4761e48fc85a6381c859fc54c851ffa447858435c8501089c7aa4799c86f4f2148aaf428c8ec988247adc546c81d39684790640ac807b98ac580c030c770db4447f5f1ca48a32fa647249312c8da53cd48c06a68c85fb467c72893a348cfec37c7c4d935c84bec36c6c69a0ec8dd6d9348f5aa50c8f8a88548056e6ec8698c56c82c66e648fddd8d479e3486c87f4f24c7e05182c8b30974c8e9cfe0488a2bd4c734e731c897604cc53af62cc873123e48cdf35bc8cc231d48116c71c8c6f2df47cdae82c80f3b0e486a0a3cc8cf369cc6291458c8ec9de8c7cc699ec82612e0479ad4d048fc4ad347a4328ec80ea789c66a7b85c8da4d4a489b8553c867b3e94796ee1bc882cc19c8c6e87cc88beb7647dded6ac828223b48ebd631c8cda85ac80b5342c8d2ac1f489e6837c8f1e5c7c76ff9d4c73ca88248e99e54c81a38d4450997b848d85e32478dd13bc8694078471a93c54835025fc89165ed48fdf82646edb94dc8e8f81f48c986cf485ce3e947cda582c87ecca9c6994066c8fdfc96c8cf495bc8932b83c8f7414ec8596e2bc8d7a23dc8245f83477f83d348038300c88f3fc848ac9e914886d0cf489d712c48e940e7489ee772469333f54820f61e48441282c844b663c88216c248378c6b4882d23cc84896a3c7d8da1bc8c55834c8ea1867c893f34f471354dd483ece0a47dd61f048096fcfc741d087c8984fb8c5530018c8697630c8320188c8a05b71470d9f86c83c284948d88c67c894df3f48f7d912c8839305c8209c1dc8f35292c61c25c94844865046c6d987c86d042848847648c8c0b6d34704d229c8391b0dc7c19a2cc8e333a8c8d2a24cc815e9ce4771e728c8069087482851b44804100ec8b9dc93c89939f7c799b37cc8bfdcee467c59eb48cc0235489181c9c898b9df47d97ecb48dc38a9c708bd83c8a7880448a58746c89fec88c8212215c8801be6c849054ec885d902c88abaeb4879423f48b3f458c8366100c8b14a5ec837931ec7e206bc4892b3b246592cca48d37057c69d843fc83b3ce047dede7ec87d72c347d03f89c821e187482c8f4dc88c5211473feed448cadcfd47e7558fc86a693d474dd2a548bcc5e247e1ae68c8dc71d0463ecb4dc899799ec74482d948519a5b4827166fc88881db47af2ebb48fc46f0c5aeda3dc89609fdc6e84be6486a586cc6df4ab24899843348f0be5ec86584d446d0e60bc84238f347c2764ec8a9987dc7902e8cc8bf5e85c8ea4bf048969fd84662d1ce489e2ac847ad7710c8515f3a46eb8144c842a26e48689b26c8a463c447bef7eb48b803e3c7fa2027c8f74b8cc57f518ac85a6ca3c83ee298c8caf674473c472cc8780087c50f4dcf48d3a5f6c7a93eca4863a61a48c839a248e79c07487658ccc7358b2648f94b36c8b9e62dc8b3e193c8cae529c861a320c814cd4946a6f9e848f07484c7cc9010c8adc3f2c744c906c874645747a1e345c8f15411c847efbb48d9a97cc7fede65c8a3e4c947d90646c83c2043477dc61dc88be68fc55b6e08c8af374048bb641cc8eea6a947150aa0481aac3bc80ff06cc833dbf047f13616c84e2f04c7ca0577c8cdc892478dca17c86b8ac047cfde3dc8d82902482e1355c8493088479b5796c8b04e2448d319a2c7bd9e02c8933152c8e4f15b47e594bf4850d9a8c7c4370549a1547647f1e552c8f8acc947eaf341c8670cadc7152dfe4827cdae477d08f2c7f0e7bc48e4bb76c863fe3d473cfd68c85fdac7c81224b648c43a8647fc3d82c8f3fd0146e3bdf24877b8af4824e40bc86a9149482617a9c8eea37e46e21f4cc85232c2484b67f548450345c809e253c8f29418c84dad34c88fbfc4c8e8d3dd48a1363a489a81cf485006f9c6f62694c8a1e08548fe6f97c8aeabb9483da638c8c054ecc7476d5bc8c63002482133fe48a9d58047f85648c8655f93c717f202c8262674c8fe1be94888af3f48eda570c8b4428ec812e846c8ec7a69c7cea145c87e451748864332c8017287c830ad18c87d0e49486db64dc8fa3dfa47a59e1ec8eaeb9d47e940ca48c5308447c6716ec8072b8e48e7bed7480eb496c7e1a298c713a359c8cef954c80716104517b0d4489ea35ac83ec2d048a98da2c782cd60c8dd8830c8805fd5487a3533c8ded072c850efd3c87323e44825476b485d8a4dc8bd90cfc76612e648fc53344852c0f248e7533548908cfe4802f9b048ee9a35c88095b64636555ec86b34f6c8bd7c40c8126542c86603c9483b3921c89cdf81c8bd6992c6be42ec4843e6cac7d023bfc70c2d95c73d5c2dc8a0d8f0476b43b748830494c68eab33c8cf3b4e47541e0dc829f052c71d0664c8f77995c74880ad48fb8b18c842ce1bc8797fcf48a64ed0483f68bec73256bd48e79ff348d0acc848714f2c47b017da489574abc8b5eb2ec8068607c820f781c89e5e8ac4b9ce5ec8721ed7c8b0e306c8c0c7c046a352d948d4c9f2c66dc7cc487e9aed47d78d064980086fc7bdea20c8956745473de3f6c771d8da48eebcf34894cb7847afb225c80552ebc7ea6e28c8b2de2cc885994bc864ae264860a9fec7a36322c8e4c489c82ecbadc70c941cc80df30ec6'}, 'ds04': {'d_in': 9, 'norm': 'minmax', 'nprm': [[-9.45396047092099, -9.99991097068758, -17.00543899716055, -10.34968380844697, -10.237159369230636, -6.887515070449975, -20.54356302962709, -17.764266611601336, -9.528392086266688], [0.05652538463058936, 0.044976980209976196, 0.027776828312146923, 0.048508788951253656, 0.04736048738859234, 0.05224044043246623, 0.02036472484187993, 0.03230028827769735, 0.05056622240306749]], 'pmean': [135.66723885313027, 286.3124605935502, 598.3598545158078, 206.9606492513918, 228.07968924251063, 191.90164345488324, 1027.7124499743848, 370.03454840280307, 224.5303778694659], 'pcomp': [[0.19506147937893353, -0.3563140073465241, 0.04760031020702536, -0.0742732249916482, 0.053375268371155256, -0.2428736283860389, -0.10396468761912284, 0.8362282219314988, -0.23513024450960696], [0.01878491492423737, -0.036218215316056075, -0.10156980800726871, 0.00031014406757639933, -0.002948282785052903, -0.028850108250006883, 0.9888126828356172, 0.09464365056646558, -0.021676769041898435], [-0.004535549256977053, 0.009939336657960821, 0.9933829585510289, -0.011314147630642734, 0.014367709362532267, 0.009567030000660046, 0.10635932536882271, -0.03640268002343546, 0.0027396646465232526], [0.18896591826953543, -0.21148605116919997, -0.024014895672757774, -0.5806754318191727, 0.6199687581852416, 0.218851308023295, 0.011330277864989722, -0.24342994116707928, -0.3002685330807921]], 'd_pca': 4, 'n_tr': 936, 'ztr': '...'}, 'ds05': {'d_in': 11, 'norm': 'log-sym', 'nprm': [[-0.03372721439225956, 0.05375075983968351, 0.0768004821084266, -0.003314235748219865, 0.05358384995176832, 0.008944045943711322, 0.055434088341359225, -0.0017343385778375259, -0.09050815082256254, 0.08306109904482158, -0.025246930871766507], [0.6026254062448728, 1.0097062229134504, 0.908077393481926, 0.5938660892352818, 0.8676418959685436, 0.7622305057808377, 1.0362885142921587, 0.8100586807988754, 0.9379326135736283, 0.844616484440711, 0.8195265969896206]], 'pmean': [-1.057355261547768e-17, -2.819614030794048e-17, 2.114710523095536e-17, 1.409807015397024e-17, 2.64338815386942e-18, -3.172065784643304e-17, -3.876969292341816e-17, -2.6433881538694202e-17, -1.76225876924628e-17, 0.0, -6.1679056923619804e-18], 'pcomp': [[0.0011106214644545104, -0.35253024025490415, -0.3350798036895528, -0.000185410148919846, -0.2892239737797872, 0.24250114223781363, -0.31678844275549956, -0.2112684856341286, 0.4469384384458773, -0.4146615060069237, -0.3229519363661594], [0.02345731070681327, -0.3561809082716769, -0.3643411285064849, -0.04104132726085144, 0.4131111372659176, -0.41303081589222246, -0.38327282056966355, 0.46432161366356745, -0.13864523793472525, 0.05877642089457789, -0.10830980807389348], [0.6204545223702673, 0.0360554183060699, -0.09908519420187048, -0.4280307563244825, -0.08772722908351986, -0.21307263187967765, 0.117224784055658, 0.06618399729811335, 0.16943084310842965, -0.317888454764017, 0.4687537983510969], [0.22063151010691973, 0.006926505031652604, -0.06264952583791752, 0.897955974094972, -0.028333109095346285, -0.15363907794703424, 0.05221672128371987, 0.06787614600002158, 0.09695371194778614, -0.19115284547252245, 0.25168024289137775], [0.7520489942212873, -0.0176490847008563, 0.11661297834649408, 0.09093046486524937, 0.05635780156618709, 0.22459661536441863, -0.111184227554927, -0.0908561556127148, -0.1672865893303047, 0.32038013357965356, -0.4553888844471331]], 'd_pca': 5, 'n_tr': 1008, 'ztr': '...'}, 'ds06': {'d_in': 10, 'norm': 'log-sym', 'nprm': [[-0.02963551175327396, 0.03243496060510082, 0.01841024790400235, 0.038561986172339247, -0.03596067971504501, -0.04760364812031202, 0.0360321515278965, -0.05067737500732377, 0.07273313288535456, 0.001863657429555629], [0.7816374688042695, 0.558849193887985, 0.9943944335512069, 0.7730798540218572, 0.8663953549293136, 0.6303686652737226, 0.8339010883508906, 0.7217818614357355, 1.0385765132021214, 0.5472149591699709]], 'pmean': [2.1930331350620375e-17, -4.751571792634415e-17, 3.1067969413378866e-17, 2.1930331350620375e-17, -3.289549702593056e-17, -7.310110450206793e-18, -2.1930331350620375e-17, -2.649915038199962e-17, -2.1930331350620375e-17, -7.310110450206793e-18], 'pcomp': [[0.14278418035352275, -0.019157029426647154, 0.4913490150529678, -0.31397136827487565, -0.4714293180755976, 0.05592744155587461, -0.3655998254040047, -0.37463845327954937, -0.37135171720766635, 0.044170463320839924], [0.545152942735444, 0.08467525905154627, -0.02203023783432843, -0.2498724695042876, 0.16794104467554874, 0.5890712827580957, 0.2699855001143004, 0.30176558399436776, -0.305430649529615, 0.016281795101769702], [0.31910747167907244, 0.17573985783104643, -0.026517726722233056, 0.5731530593780243, -0.09927926450934828, -0.24988370481480704, 0.4335376868955493, -0.37271670528559153, -0.37170119520321576, -0.02730603260324326], [-0.0641508253567102, 0.6148726273314842, -0.017105789948026147, -0.026018504062099657, 0.008077842554360445, -0.013864187923743452, -0.05728328775417261, 0.016379808721226925, 0.06333290666972415, 0.7804028162795988], [-0.07253701981221015, 0.763788248318723, 0.024562139046827778, -0.09557903504040818, -0.012461163162776038, -0.0008613307585383291, -0.09851551716297975, 0.027403100182536125, 0.05698263640000542, -0.6227101745776479]], 'd_pca': 5, 'n_tr': 972, 'ztr': '...'}, 'ds07': {'d_in': 10, 'norm': 'log-sym', 'nprm': [[0.031109160192074808, 0.043562797236322774, -0.41169693939533075, 0.14905593374793016, -0.03033590116391847, 0.14888292622748575, 0.02946013403416927, -0.04231129002486611, 0.33590194406561247, -0.3118569402359945], [0.4786736669622841, 0.6073667980170199, 0.8428961931827845, 0.754665374625105, 0.7938358414844514, 0.7191662968008644, 0.4819256779664972, 0.635434201957401, 0.9945011304236452, 1.036925820280502]], 'pmean': [1.7578531223231644e-17, 1.4802973661668754e-17, -8.881784197001253e-17, -1.4802973661668754e-17, 0.0, -1.4802973661668754e-17, 3.700743415417188e-17, 2.960594732333751e-17, 2.960594732333751e-17, 3.700743415417188e-17], 'pcomp': [[0.015836812009906266, 0.35315702853793257, 0.2767291791396513, 0.34595792903344497, 0.4440883274741117, 0.3355356385990608, 0.0012201505754287467, 0.21665116007856058, -0.3958796222898697, 0.40657703413644697], [-0.12390168246456806, -0.29663721672071997, 0.5228397038827309, -0.4099524741974741, -0.06898057807762333, -0.46752609230068864, 0.03198966338668951, 0.17217489167653058, -0.3330333828646072, 0.3005239001448699], [-0.22119900696578684, -0.38130833677200704, -0.2459721564532951, 0.24647696062972943, -0.08209106907460752, 0.06982803160068042, 0.34714045809102384, 0.740319235440092, 0.06496425237270194, -0.0027272774579249646], [0.5937312779180417, 0.04924912146125766, 0.05787769072720953, -0.0635665161895997, 0.010690982653955558, -0.03854930717300435, 0.7881603074860343, -0.11972043928962414, -0.02091019025574474, 0.009996422478489845], [0.7634979014349063, -0.20729008582996278, -0.03615213098564568, 0.051036393582877745, -0.054370690199139045, -0.03232659679013274, -0.507127904501217, 0.3283746962954443, -0.012380813644492965, 0.032046321570005865]], 'd_pca': 5, 'n_tr': 960, 'ztr': '...'}, 'ds08': {'d_in': 13, 'norm': 'minmax', 'nprm': [[-18.590164183277302, -5.848112276215706, -20.68913689686246, -7.942338920308341, -25.35619651026926, -19.480213722585017, -20.94860848375921, -15.283200365647556, -17.044956984354513, -5.501188846961598, -20.97136592895593, -6.169713062556284, -23.83595746900209], [0.026088488630741508, 0.07889456147145149, 0.027351604242876307, 0.0736216856887675, 0.022748666449296837, 0.02351126842813027, 0.021494918154816044, 0.031120655371845586, 0.031560984638094175, 0.08969120055483218, 0.022425998430612507, 0.07450915076521598, 0.023581316047198677]], 'pmean': [709.2881525220835, 86.35701730646039, 748.734747905047, 99.18476390593455, 1114.5760942084366, 823.7802064763731, 970.1030837226903, 488.4014385292226, 532.4722385087167, 64.53515193744262, 926.4291393234882, 89.73858978607329, 1023.0259839862342], 'pcomp': [[-0.01783842375394196, 0.0028121415210147285, 0.06739931327761398, -0.0005613388064434982, -0.26443324590361345, -0.2367184342723787, 0.9288347848648335, -0.013900673895580657, 0.02774745877366615, -0.005054092377785396, -0.05887317011959793, -0.0004981266169978056, 0.04451169941861473], [0.23942781453611867, 0.005377616137213381, 0.012748001081546358, 0.007827943510054539, -0.4755902728589363, -0.4486607488013905, -0.21993121520587874, 0.1214052840509894, 0.0077417742481273215, 0.0046753207498440195, 0.6063589670851103, -0.0024979619370064502, 0.2899496966690692], [-0.10341999947004209, 0.0034462166433165964, 0.03982806679211626, -0.00028180057802059685, 0.6273022046976768, 0.17658817229979196, 0.24523189944449325, -0.10957522775033045, 0.0381101319894325, -0.003157788232182285, 0.6527882039146318, -0.0029525173641675013, 0.25152675605671565], [0.3586321808873539, 0.0029177413274741326, -0.06749479856502917, 0.000628991979166614, -0.3482599562811002, 0.7978235166555505, 0.13052385475888245, 0.2294775337845025, -0.1102172025827552, -0.00822114189099743, 0.1583321719366029, 0.0029856879130918034, 0.04495905004743101], [-0.1265663009696919, -0.0016466400575696344, 0.1145318919177473, -0.004108732386390227, -0.05228029700654868, 0.11218469211011749, -0.06670683842632294, -0.08525360351475768, 0.10401931963706532, 0.001587657284252807, -0.34941927163052555, 0.0017120316418502615, 0.9004931495190327], [0.6564478033066355, -0.0058282854624754354, -0.07880854091227521, 0.003635969313160619, 0.42941755052981534, -0.24448928369106093, 0.0696922137959274, 0.44971725279178937, -0.1979630569173218, 0.0024854182000887902, -0.23031374455571915, 0.0020477248568636797, 0.13891395987329525], [0.3372243442692827, -0.0028218287351761823, 0.6368443981165676, 0.0119147948661688, 0.04017457365089691, 0.048286904385235106, -0.03219190701449629, -0.06268819984264055, 0.6734040976403868, -0.0023200728449439996, -0.023729588438992685, 0.005855693544571642, -0.13255728729293295], [-0.3985513687534824, 0.0003145806332607304, 0.616938453131658, -0.000819587495810325, -0.0047287475845489155, 0.01227820520386064, -0.027935533373021994, 0.5870727159548484, -0.3353818062943172, -0.0016150819276761646, 0.04005834149812226, 0.005865434898482837, -0.02850366773625616]], 'd_pca': 8, 'n_tr': 1032, 'ztr': '...'}, 'ds09': {'d_in': 12, 'norm': 'minmax', 'nprm': [[-17.49333246883519, -18.374743036928656, -17.09451570817986, -10.804455739887157, -14.523644768343226, -15.986372181951364, -18.837869458926843, -18.656275376520977, -17.058708520795452, -15.196697700896127, -5.641883333152692, -13.706913808711093], [0.028318818853216614, 0.02968626683966443, 0.028929567581177318, 0.040907050305238425, 0.04142335950555553, 0.029825205495873874, 0.027010365162700493, 0.02573805721719617, 0.031023771447529293, 0.03063711073294118, 0.05939543372053555, 0.045416246920433834]], 'pmean': [609.6957669501822, 611.6650991468807, 585.4272022084144, 332.9249203267412, 299.7701358311607, 532.8985394403442, 695.50177220375, 722.7519362678876, 557.9879288781566, 497.5643306131965, 148.06687349601972, 245.40650688834057], 'pcomp': [[0.0127672761109216, -0.016721620085437733, -0.020002474113734497, 0.6297776261368137, 0.6080245329017256, 0.0012933398286246926, -0.0018434837972661517, -0.06209773061726635, -0.05111424221733981, -0.005123622476473691, 0.2643747246314043, 0.39553695468561506], [-0.1763796412480377, 0.004067979112605535, 0.0792981892843739, 0.01828811298159794, 0.024139862221177658, -0.0955457678475367, 0.9001471652603615, 0.3552789798877092, -0.08942023472671162, -0.08915782594009598, 0.0019908517609264435, -0.010107256072623149], [-0.010105040998039808, -0.12079946154358613, 0.16812880863343987, 0.038931729176273265, 0.027136754918875208, -0.10080579200603855, -0.38367273562456794, 0.8908865528919522, -0.05279800005540298, 0.015030457792383125, 0.019101124422286134, 0.019030938697858928], [0.7368443992924905, -0.19908025365757548, 0.44359130387380885, -0.014281621291716225, -0.030851530451630277, -0.13452614741804503, 0.07564494797921034, -0.10232975311416846, -0.37967426248678837, -0.20250369097676266, -0.0065072023455579565, -0.002214118727451807], [-0.46000233574616106, -0.28182557724996, 0.6846779246854742, 0.01459749369261967, 0.013073444539049923, -0.30777927613249634, -0.07908153225617137, -0.22432173132607267, 0.2715719018475826, -0.12089674142740786, 0.005262353129229938, -0.010349384662059858], [0.04254443840099808, -0.5159866589795399, 0.08762430328153925, -0.0003843813474644275, -0.010777667580209788, 0.8012737769153833, 0.07874191122677233, 0.054356347170609165, 0.23865257427765005, -0.12260757646070071, 0.004715555467429797, 0.03080490239082766], [0.2775019660640066, 0.6421698240020993, 0.41006697328110747, 0.03432054722355843, 0.02070477952486546, 0.21375457394300731, 0.0672537062352248, 0.09037009770138042, 0.5067967399711801, 0.1575244034586364, 0.01952577218024408, 0.020738924435933316], [0.05692001336604256, -0.2641056221664518, 0.11685345877732319, 0.00481881841495136, 0.005290969573977988, -0.023085731158034438, 0.09946318531697464, -0.03758377623059154, -0.08121068981621951, 0.9455541385203482, 0.002404083452122942, -0.028113789097669657]], 'd_pca': 8, 'n_tr': 1056, 'ztr': '...'}, 'ds10': {'d_in': 8, 'norm': 'log-sym', 'nprm': [[-0.4781721155044118, 0.036357114718641045, -0.009436319512794368, 0.2762233974982176, -0.36686288599141126, -0.4505168965848287, -0.025367500822820202, 0.11382901272918358], [1.0309879321211555, 0.5670657638175178, 0.5747938379627582, 0.7027673961795817, 0.9470760875137639, 1.0991303039776075, 0.8334371667882874, 0.9672123681718532]], 'pmean': [6.47630097698008e-17, 2.960594732333751e-17, -1.7578531223231644e-17, 3.3306690738754695e-17, 1.2027416100105863e-17, 7.031412489292658e-17, 2.2204460492503132e-17, -2.2204460492503132e-17], 'pcomp': [[0.429204212039052, 0.0009700222788430191, 0.007967383501612359, -0.3647774968344028, 0.42539448727269724, 0.45570038077726194, 0.348931469893071, -0.4150665302138911], [-0.013079731716619887, 0.7372045886367559, 0.6740405459073449, -0.030101448294925927, -0.025490500685541553, -0.008382342257469092, 0.018464170111636185, 0.007784955348439334], [-0.07159797648489906, -0.660722426315085, 0.7206544500287912, 0.08928693316586558, -0.03841621937313432, -0.029933050562557413, 0.14044078142814306, -0.09438858952255354], [-0.3648563560116775, 0.12892223432198924, -0.1618839483194782, 0.2556812305975407, -0.2541723841376086, -0.1440820106397385, 0.6909368191845993, -0.4426315373070613], [0.19355669499694786, 0.057765846007030656, 0.001744542506545959, 0.8333667774778346, 0.45066474688820246, 0.02175538951684168, -0.1644759089030576, -0.18458425705969175]], 'd_pca': 5, 'n_tr': 960, 'ztr': '...'}, 'ds11': {'d_in': 9, 'norm': 'minmax', 'nprm': [[-2.384299007654202, -6.228215981762774, -8.507809573439552, -2.071648296512585, -7.138669988878603, -6.298335335258357, -0.9504544529296612, -4.827917776033435, -8.004031592925475], [0.18168593649729936, 0.07043910870472704, 0.07741305809424079, 0.23582711598536807, 0.10301585013683728, 0.08619362144730679, 0.3914573790097006, 0.1115231538181308, 0.07033738418193021]], 'pmean': [14.925863223681096, 86.98196786594393, 85.76646620106045, 6.748391096281144, 52.79190952129326, 69.89352625608474, 3.3185787623061147, 39.36005645594262, 94.50527444932504], 'pcomp': [[-0.09030348261572053, 0.002133185518554903, 0.5093769486057946, 0.001345208180605769, 0.24634628498320277, 0.38808702732108524, -0.011938073531398426, 0.24796343844772809, 0.6778256618663927], [0.0002587324711371006, 0.9998786458170658, -0.008223543743277445, -0.002266388694010948, -0.008781140273975738, 0.009261085532240204, 0.0003109064867448394, 0.002625009497958054, 6.298895180447838e-06], [-0.06904443097143015, -0.015296302659911411, -0.5016611318381796, -0.12754991449680142, -0.5492388086826198, 0.5823843683278891, 0.030562726565065628, 0.25560053415401796, 0.14129926765831072]], 'd_pca': 3, 'n_tr': 1056, 'ztr': '...'}, 'ds12': {'d_in': 8, 'norm': 'minmax', 'nprm': [[-8.180209340768233, -7.699353576395036, -4.245399161224843, -6.1765252741563605, -6.12754640115853, -4.666403872711895, -5.526108632343583, -7.207999031769716], [0.07766501513519665, 0.07336135262920118, 0.08215458476896544, 0.08338709391692078, 0.09443253701650382, 0.07936830173769682, 0.10695781509721489, 0.08594844733443734]], 'pmean': [93.94855699196785, 103.3794994042677, 63.403574474238134, 73.86186251049568, 62.17504476970753, 68.1298401609417, 50.00270896919145, 77.76640480146378], 'pcomp': [[0.521019840273922, -0.010211762668083213, -0.46557026578301647, -0.006439268207496652, 0.286698937868031, -0.48792703042961927, 0.18451772306392408, 0.39663717836911433], [0.0165007644258691, 0.9985806815909755, -0.026947827279110617, -0.03280699655116619, -0.020670634487322703, -0.002353422773853585, -0.01593271852871475, -0.008671646920577064], [0.009593579340792748, 0.03156678172653944, -0.020400189243648652, 0.9990474119087914, -0.013709293665458186, -0.003330980994940328, -0.011402090256588867, -0.008399686578465553], [-0.22831490544967156, 0.04117698497126486, 0.5932222704185235, 0.027794968023472345, 0.4624621455024815, -0.0685859913883882, 0.5070817870166755, 0.3431972093836702]], 'd_pca': 4, 'n_tr': 911, 'ztr': '...'}, 'ds13': {'d_in': 12, 'norm': 'minmax', 'nprm': [[-15.565056034520817, -17.681206419663845, -18.14984975368152, -5.335092497279244, -7.178671084506003, -7.689568708388659, -5.411423710541277, -21.55100069265896, -12.008531810146785, -24.67894098805461, -13.25407525768168, -13.102679625640842], [0.03105849542594488, 0.026205052935150144, 0.029334636515861764, 0.0846592086695928, 0.06994654644358311, 0.05921595394408783, 0.08100405408638747, 0.02472071354246668, 0.046134303696071494, 0.024074061566504254, 0.033281621715908496, 0.03750909525176074]], 'pmean': [512.385126966957, 675.4822410926605, 620.7155625351061, 74.85387841529314, 101.59137495436232, 141.3697223370004, 83.55745983234677, 876.7462413976889, 259.20198045110305, 1016.6833519307447, 405.3997871048258, 353.7229558272384], 'pcomp': [[0.008896664209496896, -0.12588889566076847, -0.12656341398686524, -0.0025862771932342044, -0.003237306329700535, 0.00845099719411369, -0.0047590819359888716, 0.9516846473717459, -0.036645503887602904, 0.22907886160110816, 0.07114247486340351, 0.05795138305851053], [0.015004917452086531, -0.03062454359664611, -0.03319188781476958, 0.003547008263187133, -0.005870556982907103, -0.0011183812600611535, 0.0068291729898808575, -0.24195251714443447, -0.0043528052791846844, 0.9690399381474537, -0.004429994512950304, 0.004728766777493034], [-0.21888797135048976, 0.9623226489855232, -0.10621161027189084, 0.001199206667762699, -0.006851205502882526, -0.011882335954975667, 0.011777818231651834, 0.09814546430816816, -0.02625109861763497, 0.05448864372349773, 0.03104903208599848, 0.012620540253293264], [0.6790480075223952, 0.08685768879030455, -0.5597286742747422, -0.001975073395081868, 0.0013329701187935197, -0.020506339174045752, -0.010446206356408937, -0.1021626827492275, -0.16011580928192823, -0.05219056241936245, 0.3853531548034336, 0.17384720539474366], [0.6984223469987694, 0.2159986010107579, 0.48670358580927375, -0.01447311647701604, 0.010183743771779935, -0.013844218623876801, 0.002960993865132023, 0.12281620722975692, 0.11071323374869359, 0.04308808588377772, -0.3978400328420155, -0.20169859632759135], [0.04211756109912145, 0.04745661607781615, 0.5274187471191372, 0.0021588648618510093, 0.015922078239938967, -0.036105682181484945, 0.025755128783178375, 0.004021823315959092, 0.2107048517266645, 0.019587895399439236, 0.46791878610214754, 0.6721859415146656], [0.013678864209735814, 0.01006983920096248, 0.028225650299244596, 0.011546253022574908, 0.001429063113024965, -0.00542609572969972, 0.0009570457168149525, 0.020469846871041548, 0.6030207997278776, 0.014172176926627717, 0.5375203186864814, -0.5878495044745199]], 'd_pca': 7, 'n_tr': 984, 'ztr': '...'}, 'ds14': {'d_in': 14, 'norm': 'minmax', 'nprm': [[-6.881452270538005, -5.322140183153492, -20.50092065512895, -26.811479852245533, -25.9843999499914, -22.0273705543446, -19.25138643900205, -17.19374348261863, -20.336068972058328, -21.528465741388658, -5.5322707486723415, -22.04320796440215, -7.097450650941558, -19.87656511438471], [0.0684428688903544, 0.09052921401028342, 0.021574434400079803, 0.02049536363587971, 0.020434192986335604, 0.02325154385328339, 0.024423091888136447, 0.031000724709140646, 0.02368106846004016, 0.02536021023798539, 0.08066054613989138, 0.020110587315817323, 0.07555853744754035, 0.023059632070334967]], 'pmean': [112.48434902027873, 59.53983082090772, 954.4313488403361, 1327.6427968657924, 1277.2747862616957, 926.4424746076685, 773.9119968710719, 561.7733457313105, 876.2944667809252, 830.2305001606056, 74.3650612447349, 1082.2284677959997, 103.15569174831879, 852.0468556894956], 'pcomp': [[-5.756815662099371e-05, -0.0025396813963121058, -0.3620730647420777, -0.25083053740585876, 0.04885319935104214, 0.053263852266235104, 0.024881143061814083, -0.09382564507503616, 0.04974304661684852, 0.5260083511146826, -0.0002649842951523763, 0.7141967583334862, 0.0072423281353588935, -0.04523273948956139], [3.2482370884462787e-06, -0.006758292847493967, 0.16838411576299175, 0.8683682718957766, -0.2308478240688659, 0.09180713193318579, -0.11397847849611148, 0.11357360094377847, 0.03205594324429486, 0.12653457532884402, -0.003598492338474169, 0.3274922967167816, -0.0017747800581165195, 0.0749829980516433], [0.00587563093436128, 0.003186135655527389, 0.19029613804211803, 0.1623105550426154, 0.9516216187138581, -0.026146283703980214, 0.007450157794188561, 0.05236161832489094, -0.04185228376992, -0.031251072534667156, -0.0013809416146037725, 0.12886973319394673, 0.003015077124755717, 0.09479853782751052], [-0.00611218404561069, 0.0018163956687482496, 0.7551751461226711, -0.33991070670192713, -0.16878764507359498, 0.05857950145049717, 0.16474335106891802, 0.2865594200540374, -0.10703703883754459, -0.10566072797305259, 0.0037322980700608397, 0.3876719300414398, -0.004240420003331977, -0.0032401121146996264], [-0.0006832572145173344, -0.0003063026781934264, -0.02736323803133779, -0.1445191390802499, -0.06429843653746015, 0.03587585508653764, -0.37660018640697285, 0.009418507497334817, -0.15056119727633563, 0.0066503090748414725, 0.005269112874347273, 0.013959833593842869, 0.0013988261649092056, 0.8989380615050359], [-0.0010596963952528772, 0.0005350272652366178, -0.08436384215253698, 0.014601328818580955, 0.03730576392028544, 0.9020412960368013, 0.2122434312145195, -0.05548206609527275, -0.3480121433713514, -0.009729948777763571, -0.0038673194383029253, -0.09081336929836392, 0.0015898745965061464, -0.0008395550587323166], [0.0006080816365198458, -0.0011336767366898343, -0.054914595074523335, 0.06157354623038894, -0.02090710418304211, 0.018063951612142665, 0.7430236853408537, -0.03239885142882127, 0.521604453615657, -0.05100712900324857, 0.0006251754631029242, -0.009263493972828376, -0.009132751056685531, 0.40552631383668514], [0.009300558898931268, -0.0031544217402068586, 0.12517474799919934, -0.12238199963121735, 0.05145214672760757, 0.40995054489276844, -0.46808714790818867, -0.006943687947355994, 0.7508685095649984, -0.07632925902973121, -0.006634044381461333, -0.0006230856664349552, 0.00885972311664931, -0.09820562327416887]], 'd_pca': 8, 'n_tr': 1080, 'ztr': '...'}, 'ds15': {'d_in': 9, 'norm': 'minmax', 'nprm': [[-0.7779297073275476, -0.3922157118378345, -0.609609627945825, -0.6566394081173907, -0.6731896836240052, -0.7096372230623598, -0.6250594322345443, -0.6589270724470789, -1.1662708497184835], [0.38590885899594196, 0.7200233136753941, 0.8357917813765045, 0.7791006535393389, 0.7633794099172313, 0.6577577216310259, 0.8040007334144395, 0.8197376819779931, 0.4907795848931755]], 'pmean': [3.2148486961621785, 0.7728167970637243, 0.7413159319896135, 0.84904138374399, 0.8937780236808855, 0.9878683607342473, 0.843168506509644, 0.8541961171241315, 2.173249480475531], 'pcomp': [[0.8070835319506601, 0.16502278233503398, -0.004197249278089187, 0.0018359055232487333, -0.0029931582028093757, -0.26264524997332034, -0.0697526303759309, -0.03454296050165969, -0.49629883580329276], [0.47202389306545495, -0.3633212733810782, 0.007691857296679956, 0.010439567968175282, -0.02522740662938095, 0.07973426519756578, 0.36785148433408665, -0.40597972922698966, 0.5812871212010455], [0.2804905447045098, 0.44755394823823097, -0.025324551347898044, -0.04600147025804833, 0.019175723629471994, 0.658062069771503, -0.421342709556614, 0.11134456990348963, 0.3080947279469294], [-0.010573341446553731, -0.02885284964284391, 0.10731921346620102, -0.5074718620438817, 0.844315275619259, 0.05294117686360326, 0.06361729343108637, -0.07680986366158694, -0.06627703329144997], [0.017435019604060725, -0.009180049093737912, -0.04843026587493865, 0.8491548876969951, 0.5231856386959782, 0.0032218148421218253, -0.032599942006757296, 0.026280531220930516, 0.02674355378284037]], 'd_pca': 5, 'n_tr': 984, 'ztr': '...'}, 'ds16': {'d_in': 11, 'norm': 'minmax', 'nprm': [[-0.9753524370744102, -0.2865772355267929, -1.042063613754696, -0.5217250561926874, -0.9912920331481702, -0.4620367509017783, -1.2290742195329447, -0.6723431235533581, -0.6374716548284871, -1.3139062382107232, -1.0566676789100444], [0.42105570734314396, 0.7714605295645974, 0.4070086998509457, 0.9647817339420196, 0.40385962677638904, 1.1263351627908065, 0.5053700304162433, 0.5849866104692987, 0.5448478899252331, 0.523759868276073, 0.4706720999731647]], 'pmean': [2.778541034688433, 0.9385420844585391, 2.745819358341255, 0.5488097172044786, 3.0302657243391375, 0.4096683823841886, 1.6954170400710475, 1.812244023287252, 1.797227073371455, 1.9726888311058788, 2.2835025944517433], 'pcomp': [[0.4992395749124702, -0.07069975287274255, 0.4896198031127247, -0.0032704622609145643, 0.5500848300671337, 0.0022422589974284085, 0.10741811864652677, 0.18656352662805195, -0.234715840431013, -0.19278199964829557, 0.2546051245496033], [-0.2406160976014467, 0.10610713381124463, 0.2924113367124767, -0.0052493925255912306, -0.11014717793664307, -0.0010644534586239562, -0.5059465928930716, 0.15477378953940946, -0.08140759551817961, 0.47814327104034177, 0.5639101212633709], [0.15143169904299822, 0.4015491741143806, -0.09327059638125977, -0.016652156303323645, 0.16347060631437793, 0.006694203917364077, -0.51247845413976, 0.3516619035749603, 0.5256385651998313, -0.3082784749040651, -0.14983581561599146]], 'd_pca': 3, 'n_tr': 1056, 'ztr': '...'}, 'ds17': {'d_in': 7, 'norm': 'minmax', 'nprm': [[-189.65272186421927, -257.05870071244175, 1.3318005085489726, 3.174656096702161, -82.57358181018178, -265.29027155068906, -262.69407585414984], [0.005331095422915744, 0.003938643077445392, 0.009371474842223408, 0.003537287732540642, 0.012267184747283026, 0.002408659357655765, 0.0038589093277995824]], 'pmean': [14524.585573537735, 26676.24170375189, 6696.3715001869905, 46873.426407937004, 2734.080538876036, 111713.23186055814, 27809.110823043924], 'pcomp': [[-0.2741306615120953, -0.5013916683227636, 0.08865997967758894, 0.6219480231781843, -0.05179926921119636, -0.055987435582038725, -0.5224567758101224], [-0.015304532251400946, -0.028120078115632812, 0.004940713446554402, 0.034910235547661386, -0.0028842063455808502, 0.9984314698349362, -0.029294687248501428]], 'd_pca': 2, 'n_tr': 888, 'ztr': '9ebc4646b5f1b2448a663446382d8fc6c0da914649e6d2c5c9f83ac7c7c78dc5060a38c73d074cc6bb908d4663630146f9958646d4d82dc557e4f846e1b5ff463a3cce462b4700462f0946c7f29539c59929a346ef8c08c77eeaca467fe30847a79b014765b60247124a6b466ba9f544eeb952c7e3744d4610592d46ccef01c7766e1247876208c7f320aa46c7d880c6ab1aa746fd490047fff2fa46860d02c7b1cb14471ea37547a0142dc7605590c665db62c7a5b1e7c2a99b1cc7248697c5cd83f946444b16c6588750c7c7a548c787e242c7b77242c4d91beb46dbb12ac61b9205c70cd98046ade116c7035539c4f5a9e546c7084246ee1ee2c6738e01c7f7603c4667015746a885964683e8a5c634fd57c7b1130bc74a4e1bc71c1bb044c96953c6e32a2dc745b41347457e1246542c234608b590454b9316c7e34f27c69ad644465106a7c6873e22c78fd384c6a06f014624fd5ac578b230c73b912e476b43da45dfb606c650cf714624ced745b73318c7021cdc456f40b54675d71a465f1152462c824f4638af27c7ab51db465ae26ec7942209475fc8ac46df8c3146640a3146c0b039c5537d0c4752ca8646f8d55946671f27c6aaa984467311e0c65d844145bcd78945da3e4dc7006c02473a5d5dc71d970fc7629576c7442f264659fa56469004c9c6db90c5463546c7c65e1917c7e9859ec694e544c7e13732479131a2461879d8c687609b4651c138460c8ec646314d2bc42ff80047bbac90463ae865467494ce467619ca4681a0d1458f41e546a54cddc602fae646c8919b4621ba16c75438be4491960047abb277c6f6070a473fbd07c7996150c7fa0584c605201bc7129066c6b468b646ab8e224717d232c76d3d8b46ff1b47c7e66f86468a6caf467184b245ab8539c791869b44ab298f468699d3467b2be846b1d5abc69b4b8346ac3d79c5ae67ad46278614c6f301324657628746b49fc6463a24c54468f707476e7f21c7bdc470466603b24517cf0d47cd65844647c500477f3b55c685dfc046f695bcc5bdc4d746b94492c6808e18c78225a7464370b146d3170fc4f1d75a465fc25d4664b3f7c6e0412bc63e0bd745e9734446b3a4c2461eee2346ca5924472f7003c6bba3e946c983c1462c37a84616b0b7c6005abe465d7406470cecb64691df90c538c363c7f47a9ac60da08446dd7987c6ddd2d046856774c5949ccf468d580247d12e7046db46a0468c73d646fa8653467cf050c70b3947c793d506472c1df945d00f78c5dbd74e46078e9346bf7cb846ee35edc6dfacb145e3782fc71face346f62a1e46f9bb824665b59f4652c02e47c299334672638f46734a4bc77d82a5468a1b2b4729422646f560fdc63bb940466f3eb146ab549b463739f346cf9335c720ca75461f59b9468c743bc73c16a245c8b28946d7ef3646226ce0466ca811c783a3c1452d3f5247810738c7524a99c6e0e075c768229346cbb756c71f36714607c52cc7069a1345af4538c73b19ef45660f3bc712bf3f46c556c44697fb08c7730113c7c624a0c5cb8d4dc7311327c59bb6b346d8484a456682de46fb58154703eb94c7dc727946d94b0145049c96c69d3e2447a664b4c63775e446fffb574671a53a46dc4d8546327aa8466c2ecb4532d58a469d65bcc5d1f90cc7ee32f2c5be7b554618e8e4465e6635c73a39084752421cc7b1990445f33d8a464b2e82c6c83f51c74991b746c51340c7d09d06c674038346e82c30c61681c04605391fc6ee050247994618c6a5f829474446f3460dd862c70f6e93c5d7fd3ec71bdce7c5399bb146328519c7b7e231c731260dc7d4969c46b7e054c7c49ccb46f9e915c69937a8467b6a8745b7b1eb466fad1ac6b82847c70b6f93451be43ac7486170c62c4b754663bc16c74bc02d47ea028645b98df746672b4c454c250447d1ed0ac669e19546115b6e4664912c46b0bef5c436e1f4c405785f4624fa9a469d305b47df37b84606b1e0469df026c73d04634679988c45f93f83c6799d2ac7c14022c784cf76c70d752746fc68cd465ef247474f18ba469063fb46c3d1f146250265c6702cfec67c8c7cc6ab84054755ae4847e986b346b4f6f2c6c652d146b5781246d709884654e32946f76b98467f85a0c6b7ef0b47475e6ec7b21c0ac748dc25c7db96d7454120ae45c8d44dc7ccb15bc42561b0463883ad46830a0347f68586c66a4d0447e24110c6977eaa464555844556eb004710334fc683c62dc7ab62adc6db7ffac6a8866046a3fb9d46e451c045bff69646fa514cc6ef9e9f466cdf17452ef114c72594304529be1b4788ceb2c6ac54d246f3ed04c639dabf46e213d8c5b8411b47652bffc6b80e0147a3e71d46a4c3b146220e9746083c38c7c04f2bc6cff4b646474aa8c54a3d02476f8403c62a5a8f468cf25d46632765c79efc07463b8cc446f7df95456e98944644ec4f46e67406c7e366b0c6fe661f474d8f33470e6f9c46d4cafdc5e7e3c846a285f44611a35746e13d10c609e18046896309c7f189d846f0b16a438f3ac945e046e5468647b846d21633c67a32a846a279ecc5172e9c45ad712747c1441147821fdfc5125603472fbdbfc54683944594f76bc5bbd1ea46611071c7cf561bc789d89246661c19c7ca376ec4e0ea21c719651dc66274ce468455ce462601e846aff69d4690ba0fc798b7f945f0650847fde61ec7c7434bc7042fc646c86d44c7877388c68f8ed146ae7812c6eaef0ac7448a4d46a3e2b946cde7a146687f6d46a8a76b443efa8a46d4b132c604079446070aaac6395ab1468e579744c798bc464798b046f13ca446c0079ac6146744c7594a14474fe40f462c1e87461c2c28c75e2dd2c6f0001f46e524df45b3570cc75656bec60682a4465f6b0dc4bd93ca46916cafc6d9241dc7863798c5411fac46517852c699c8254723fb0fc752465846c8acc4469fbccac6cda14446dcf6f946532833c66d1dc146f7b8c546e72131c73301f146841959c759644245454f6ec73d0e7fc6088b47c7662af94306bd3dc73b62704583912d47f46c0c46521cdb46a0e72bc61dc84346787c8b46a82e30c77b710dc67e92f246b3782c46179d0c473a57db463603e64691e66d4651e5ec4658621c46eef0de461b25df4567dfab467b8776c660967ec7e15004c58a1951c7e9ceca46ffb7fe469f2e3546706c8846b0563b46cfbd22c7a7a6c746a60034478e7333c6c2d5a946b93c59c5834d4ec71f5f01c7b2093ac739189bc40e6887465c0ac346b7d839c71f83cfc56780be46376f9ec4ec5dda46ab5403472a6e4cc7964fb1c689a7024723ffbf44960ffe463f7807474dd036c71a5d39c6f837ac46ce32a246a12278460b0d0045ac42cd457a927cc5c027b44661d35bc5e9d7c246fb09c74541b4e94622e8f54529d6bd46a12d30c50aca2a463c87a6c6d430b346f5af8143c1b28746e46669c6de096b46c06f51456fb5f046ce964cc6e200d44629657f46c3e3f8460b7414471c0728c7bff599c65de18746934e2e45a4ef55c7f83457476c4bb0465953aa468e5bd3c6a1b103c4c68801c7c85433c78e5feb46d2ddf1c408d49e46f4524e45159bd1460d52f2464829f946ba18dbc60d586dc707f27ac62bfe21479238f8463fe8d84692e66c46c1de194703c2054603e89646588dbe46102832c700865d46ee1b9d46d7a6d646d7f01dc72a6c16c7da100b466cafc346eecbaf4648e2fbc4432220473a2e9f4651522a46cad65e4661b329c7cee59fc53e10c046766ad2468d5c2fc73efa0b452cdfc8c68e56bfc59e55dc46f61a6fc680ae50465ec009c6ee9d9846c51fab46cfbc84c7e4b198c6d8b1ee463e4e57c5a24ea446304d5946f32903c7cc371446f54514c7c8d03d4680dd6746a1fd9045235daf46c74c4dc62d39cb458a285ac7763d5045d0d29fc5e3e608c7954a374610c11047aab4b3c6aeb98646f5b8114760a93dc7e0a46ec768590547161a08455f2b8a466a6fe346de3438c7877cf243ee60af4685c1c2462bb8d846331728c5d07ceac6f7e06fc6377ce446d51d1e4519f6ed46c989c5c6502819c73e1b054603042c460a05e645e9ec00467a0836478968b546bb5f2446b0b67b459413fc42b88f3b47497300c6649ec546ee475bc6eb0095464459ff46281f64c7b711e2c63a92bd46319455c62a75b646dddd0bc7b955b94642cfc6c655922cc7599dac46638f00477efc83c69579bd46731d9dc64eb25745c4d89245828fe74657546f459b74d8c60bf1e8c6c8b030c710c98fc6c1432dc7f7d6e7c5f601cc46609dc5451a7e61c745b29346d02b8846aff7e3c4401d4c4671368a45e78b24478e2189c6b695de46f86bbbc66d4be146c8a9c7c6181257c79147a5466a06af46c5948846fc341947c0d478c688d17546cf9a9746978764c75b1be8464092af460b979946cf36a54659eb83c65ae91dc7e86927c6a5bcaa46dcdbccc6526727c78379af460de540c700ece4468f866746f62a48468cbd57c738e79846f3a831c74444854605f2cd46c94532c6135837c7441517c6e21018475f102e46b040de46c5740745d81e394709e12cc6a87167c75296a846330b0bc728918b4438850a46b938fd465c4af045fd3816c7c54baa46d447d5c531a28246545789c68b67ab4626d08044b7ddfac67844284412d9e746bc7dba4641113ac6c63e03c7b04475464ba6dcc521f73ac7ba0601c77a3c3ac7f94cbac64b869e46fbdcb2c520bf34c77deec0c6385eca46254603470ee5d546302b2746664cc146f2144445ea3d33c7f8d902c7e93e92468852b745c8b6df467e86d2c5affc404672bb90465d523ec7feaba646a63c7ac7e64a82455158d246306e0246889942469f1b0d4768dc3cc7a751a7c531a205c7cff4f9c537735dc73119b2453bb548c7cb0e9146ea68b2c4de01ec45626909c7e365d846298575c79beb01c63789a94633de62c6b6c564c7d3e842c67f0667461704b6c64b318646a0af7ac61c4946c79392b946e6fa8f465f42b34540a4324611adf3466ba1c146cc3c07461799ee46f0c7d4c516a99246277dbbc6de727c461800d0462bc10646fd70a8c4f2da1946e5c62246799381c70eabb346c972ae46c7d86cc4d97856c413d2c2452323464627479bc5df2a29c715a2d3c3a2b730c7c890b645891ff646820141c7da574346a532a5c6954c11c7b42fe44565950b47b4a627452538284614b105c60d74ac46d8ea4645a6bbba46266dec45d32969c7abd2b4c4be69f1c67cf24b464eefc246c24d07c6a6bf31c7a9d1f5c547f816c7abc87b44ad95084693d089469a3c3e46587d2f46c8ea174722799246ceb02a47e587ee4659f2ee456254d34664c6de46ffd64ec44c05f746c810ef46975fe1463e3c46466ac316c7a3505fc58a033bc7277c4ec6cd7fcd46dc2273c6bb575ec70ab9854520e652c7a8ee414659486bc7f493c545b99ec04618fc10c7a3e3914623cb32c6cc13c2c65dfeadc4260448c7d9cf09c62e78cfc443cf2547eb342e464b31b746928dd446bceae6c530bde545c6d5e446cfd2f446e50b4f46bd2d9046b92b67407d109c4609ba1bc6c1ebee46cf49d6c5d74817476fb885463b6d4dc71b48d4c5550b28c709ea8446d5bef9c68772594685ed13c797902f46b35de64658a328c70dcc35c7522099c69e8d0447ce7381c68e1702c78e2838466022e646290738c7d79bba46f079b0c612929246e5aea0c68f8538c75af7a5c692f7e446710f48c65e2a154629b01dc510eda046cdee6cc5f12ccd462b89f8c5010dbe4555fb6246ea029f46aa9341c71e0fd346e6262fc77f7b2047da8190463a3f7a466d7ab5463b8f16c7af79a3454fc61b46f688a545beb1234667afbe46347da246f70c5945dcebd5c61dea8e45b62b41c70d84664619930c474cd906468240a2c4cfa27746ec06d946155c9943d2ee68c77dddb6c63702d94664d05dc619cf0647c1c0d6c7fcae5d46ef9a98c6a0d3c34683fc14c6abaf5dc7f47719460c5c28c77c6fd34698d034c77fb457c6b7640047c47fc8455bf0e746d7bd23c6dae690464409aa44526e4cc793baa74614f334c71e8ed0459bf89946ece453c60b340ec7cfc454467a8c5cc5eda0cac67924c646fff487c6bd9f104771768e456379c546d95a2b4564d94dc797fa16c55eb059462adf24c5eac037c7ba3986c6305a484695f9d9c54f0a13c7f5b314458ebe8c462cb733c727f9cf46666c1fc713d245c734850cc74d227546635a93466f0a23c787230647979dc9468527d1c5dc06be46bea96743cd9e1fc70a9448c6cf88e946999ec0c6dee05ec7a3d8a5c54efac0468264b4c64fa8fd46dd3814c5ce1262c76774ed45d682a5466d81c5c60276be46e29d0cc762c52a46a9a12a471981e1466ce134c5dc15ff4635741447e832ce46804f98c6cbf50847d612a4c562860647bbb29446e55c2cc792a1e1c463bc794637ae0746405101c7c988c0c4c3bb48c77ac41846b65723c7c3dfb5437f4afe45e81a6a4366c1da46c0b2e0462e6ea04657e95746a32b1d462a0a554599982d46c578c9459fe709474d2014c5b7b6b64674594a46a8b1aa46839658c572e91f4556334d4698e1d346600b47c6730f1f474f2097c6aefc08c760d1da45cac01a47e8a880c4900063c707dfca46918f0345a43069c50f6bd7468736a7c6819fb6468e14a346f85e1246a2085f467b9326c50fd31247cb0f28c7486456c686192146979b6dc639bb3f46a948f3c670c2b6463eb80d4696c648c7921d08c689c278466e253046216a7946cf985d46214ed3c6e5059346f1a63e464360a5444b3744c7aa3ef5c42f7e8846257cc1469ae0a9458dd96346955c29c75d59bd44940559c7487a89469cb639466067694525ecf7469fd8efc6a588d8462de307c7804915c7b797e0c686c612464125864553e2d946c74b3b460dddc846e2562b4630b43b46521f23c47a70e246ad1340473b3b0fc75ec09146d439be46398ba7466621c8c6e9bed2456b4cc546e2f407c7e06e44c736b70a46587556c72c4222c619c09246aa5dd3460d0650464d0d55c629679346da17194583599d464fef06464c9607c7f92ba146283fb44685db134558bbac468af63a477a9a394655865f46583c58c78a3c80c67ed0af46535d8b46f3d1e4c6b2d56344ee1d46c7b4efa3c4fa5619c7de421ac6ab61a346fbf3a64543daab465ffa8ac6137de3451692ee460efcf4469cdfbac5fa5a23c7cf062b46371343c7e9bcd3c6e5d7d446073f46453c57dd46b8d3b4c4b79bb7c66bdbc3c60d7f4e4644f57745ccfd5746f6334c4628d4ca468dd62c471e2e884381c6b7c6e321d54673e9fc46aeda2ac7575e6d45eb0bfb455ccfa8464d6c2ec7b2ef85c6c4058b46539af9c6a2678146504b14c7b00beac6067a074675020747080eeac6063e38c7e11cffc4145e2d47cd8402457fb86345448ed5c55a7c45c75c3cf1c53207c046cc326246505e39c79d7a3cc2c4b828c75dc70f45a9f44546488983c4591d8746cbac81c6601a0047d58a09465a8e23c7e69b80c6f5584bc750b1c44697bc3447677997c64341ddc6109140c35ba798463eeed4c5b25983465618cec60ddc48478bedf246c5e786462bfc464782dc91c68a239dc69df016477d19fac603dcac46e8e160c6d12bb946316383c664a487467908ed45ad0fb0c6d9c9d8c6eaa2da464f97c146392738c7af6ecd46078f94c682ca14c6d65409474ba2c6c63b927746485a8ec62d75f6c60b119146e48922473e4819c50704d746019b7445e2516446ad6ef9466d166e46fbb8e6c651ca0c4723c9f54687070247850c29c63a53e046f08bd1c56c8d2bc77c5e1dc6d5a04cc77c1dad45eff418c542016ec65250e8c6658aaf4643f6a7464faa8dc635071247caf096c60631704635ba2a46cd13c246ded13ac4cc36c24603cbbd463488d8c6f451b945d59dab4602d2fe44a2555a46ef072146d0f14946ed4be9c657582cc7f842a7c6c08d9745fe560346264285462b4887c6a6b4a146b53d53c51d49f9c6466f054419b5c346ee63b44678c6fb469719ec45baf848c7c92396c61b0bb8469b73b0c584d83e46c9c1a9456b57f4464d8287c62fcda94637db01473d718646e293d2c626590c472f874bc68f741b47d281c3c6932f5bc7bf0818c696f07146af3780c562c128c752bedbc6ce8056c77dca1845df092ac76e49024654a06846015ebec61fb32f46dca60547773346c779235844e1eb59c7ff024ac610ab38c7b64290c7101039c7640a8246a7201747b38c8c462bb47f4638d0d5c614decf46d83237c53ea114c768a186c6fdd0b94694c5f146ef181347ac80ecc636f0c6468b3ec1c5358bf9c6c63e87c69df05fc7f46921469e6f5946d717c4455caa0246cdca03c5bd227ec7604163462d3f44c70f7f16c7ff915d466d1e9d45838528c7c72dedc4e8ab174708784fc6485ca6468804a1467f3d93c5c618bc46ba3e97467f7ccbc42a091146cb6d0ac611199b465c546b46ebe2c046f8f8a6c6c58c8e465e866fc6c565b746761f3247a82f0a4769f824459c719646a1b948c6e52f05c791ef5f45db891ac7e10e5546a1a2cf46e07d0dc701333cc7b16311464cafd44670b570c60472c5467ec68d467d0a0647ccd69546ace51dc797bdd6c54d934ec7dc0b10c5dabe23c70bada845af1a2f462b0fa945eb9819c733d1564522168246cccca6c6c7c3e1c6f8c0aec6e11e3dc7bdae10c6a3f88746c677724659f805466ee4c246890ee0c6c6925f4285893fc77923afc52fb546c703a894c60e6e054677a3bb4699ad9f46d1de8fc775b33dc71486eec3f0252ac738b05b46d2660ac7791e5046a52a4ac7a65d99c5db0d05475b1671c6aa8023c77370b64596adb2469c5db8c6318df8c64617e2454e3c5446a4014e47695f2346d68e7346ed9815c7234863462634d046a09882466d3c0c4737b2f5c520f0d64668e12646e3eed1c63bead045a401ba460422c9c555aa1ec7ef7103c5c0faa146241ef0c4660cd44604539a450fbd20c7f67f8bc62c15f746143d9ec6fff585469ed3bf467bb2b846a5d90ac776f6044786bcc646e93a41c7da0732c77337e2469dc39946b71877c740dcfbc55c527045e15f14467acdf1460d3489c61e6b03c76422df46ea00a74628471e46afed4d46ef802146fe7d7446fa1330c46ec3b3464816cdc42c0cb746d3baa845ecbb40c76e3329bfeba83b467db7b9c6f205bd46da83e2c6bfad5ec7cc9ffc469dadff46ca2fadc5c2b4e74624c82ec69e2ac646238d1fc6a55f0a471668f745900b7b46fac6f8452e27e6c585a0ee469af4954675aab046da4c4dc790a5a346b0144d46815249c5f982e74690076f46e83a6b46808f8f469e61874688a4f4c698db1cc7227ab0464976784667b81f46c3429946bd7ce7440bb62cc7923dd8c6b6608046c60f8046c8658a4637d63c45389b1147ac488cc692d020c5650d674643ba82c743c0fe468a972347935bb7c43ee1c4460bd905c5e7bf0e47432b9ac694570546a265c446e05311c7b837aec55f01c1466b24d5c506cf9fc6578035c727d627c7b8a81746441730c7b96cbdc68119e4468bbae1461603294701ef1746eb9833c718e953c6984234c75c856c4529f74e465910c9c6949c8e46852b2846388e1747ca5e54c79c23c64699ae89c6264d9946027dc146955e6fc7707378c638319c46246f6bc612129046e860e946b1cfbd4697e16ac70b4359c7a8d3a646220041c727966c45eb121ac7650fe2c628c10147b0b91bc770ce2dc70ae291c6d2c4e4463132a9c66b4202c73ba1cfc6808330c74a2a85c68ff6274637cccc46bb2f60c78e727bc58dc7bf46c3e17ac6b8f9df466008c9444b367d46658d1ec38d5c4dc7cb4b2b461e5b39c78fe3c442a3fbd44676a4ef453a442046dd803fc2f7fba446ad3a4f46af2e0c4722cd864594968946d0db3b4669390347f5714547'}, 'ds18': {'d_in': 8, 'norm': 'log-sym', 'nprm': [[-0.0011965781730841627, 0.013320625041211447, -0.01650526759507731, 0.037320483698617984, 0.034760933398951, -0.005840502781604983, 0.016261963289600105, -0.025518840181226788], [3.2521046349366713, 2.096467892431885, 2.6494000778898084, 1.8219086849400399, 2.0076790187538314, 2.4518223654111377, 2.667586511646611, 2.6587435590540696]], 'pmean': [1.1386902816668272e-17, -3.795634272222758e-18, -3.036507417778206e-17, 1.3284719952779651e-17, 1.613144565694672e-17, -9.489085680556894e-18, -3.226289131389344e-17, -1.518253708889103e-17], 'pcomp': [[0.37990132380032593, -0.3808619299606506, 0.3817588416410048, -0.00586499063850172, -0.36347724757924416, 0.376059963429375, -0.381669252951543, 0.38162381862806016], [0.008508654603245461, 0.002490473219577044, 0.006371523160041117, 0.9998880845772721, 0.007149484812568786, 0.006654307470939486, -0.00023164762879529876, 0.003028820129093569]], 'd_pca': 2, 'n_tr': 936, 'ztr': '31fe3ec0411f353f19fd3ec0bf475bbf541a51407b63943e651337bdd4298fbfb39f0f405eb88dbfde5f65c093c3fd3e169b96bfb3bf923daadb62404bd59f3fc6253ec0756da13f2f6110407b74613f4cad02bfcfc630bf88fbff3fe0752d3f9ce52fc0e3b9633f6749484025678abf94433c400ef4bbbdf2898c3f6afc90bf77dd40c01dde213fa8d3564012052b3f4b37314045e44ebf103a59406a9cba3df1e34940e878843fa723f93f2c143ebe623552c0918f533f83fb0bc059c46e3ff7954840244e83be15e329c0839d6abf855f46c0bc9faebfee8530bf6c41923f691626c0a4b8ef3e95071dc085541ebf65d7d73f13f1febe6f402cc08ce4a5bf863f6e4041a7c1bf1fd34dc02b04efbed08bb03f77bd8b3f33f8a4bfe22f5bbf026d3840a048afbf095dfdbf45f4c9bf8f4988c0e33be73e9b6097beb6fa9f3f3d744dc025ad9d3f1be357c05895c73ff75a4e40f13a8cbe94f950c0446996bfa61a40c0f9e4a3bf54ee4040491263be63363d40b8cc37bf381eeb3f2f0b54bf9db96bbe92b1b7bf419d86bfe0a2853fda4173c03f35a9bfd0f7e03f463f253fb3fa3ec026929d3d5bb00bc0fd8f813f2fde20401b332e3f1e5339401aa9a53f79d0fdbf5894afbfe98074401110953f92cd15bf05ce0ebf1eb5f3bf84a061bf5d0969c02af19ebfcd6be13f272fa83f98876440991893bf4c1f81408a4c9cbf38446040c7b1963f1ff127c0c03c7b3f03428bbf4ddfb0bed575243f2cfc9bbff42f0840d9926ebed6bf60400009763feee70240221c973f711e11c02ddbafbffd6a5e404b96653f622b53401d54783ffa7111405dbc253fc47b1ec0ef6fb6bf56275540ae44ac3f3598adbfffa9edbeb8fb8ebfcfa87f3fc25f35c0b9fd4c3fe78965c0dca6bc3f4474893e3dfe7ebf1bd424c012bb6f3fb3951840dbef953f482a88c0289e8cbf726cde3f8edf5fbf63038cbf287a833f1e285140ea538e3fd7eaa83f785f9bbca8e349401d81a73f8ee020c078fa61bfa6d80bc01a56a8bfd58b66c0e56033bfc8cb2ec0f3a4253f5d0f1c40c87ca13f441348402c2756bf4bd40ac0936f363ff860604049f4443f817c7c3f9f4584be7697c2bf0bbe113f94ab22be76e0613e0d97d4bfc3b16a3fa2283440763030bf045d25403c9214bf0d994c40644cfc3e38f529c0a889a93f1f3228c0570a90bfc3286c4064cf7c3f6844f13f1db98d3f03f200c0d85cce3d1481743fe883bf3ea864fdbf381ceabdffc46ac0eaa3983f3f92cbbff9f348bf4b7f53405b133e3f21605140dbdd96bf51ee1140dd0e943f62f2dcbf458888bf72406040d51362bfc279a03e913c26bfa1df3440f5a2073f40c15dc0f9365ebe8ca1db3f5ed91fbf7c441240613e2abf5370a43f13d2e6bec4e2133fcedd8ebf1a5a3040959d7bbf1e30fd3fe6a6853f8b326240609170bf5d896c404b72183fe6404140adb9a9bffc55eabf41ed00bf98ade53ffeac97bffbf33c4087f4df3ec43ef4bffb10823fa1e011c01060713f9df3e53f695735bfc1553a40933e3abf16db12c0cba8acbfdb7f47c0d7c1c73f72d103c0b5469abf9c2438c0a4276fbf8b411a400a1da83fee9202c0820b593f4c310bc0cba980bf17e62dc044b373bedcaf45c09e5036be3f944abf7efa693d2abe2a40e0aa963f975ab03f4672ba3f10a64940430619be1fbcdbbf2d54a5bfb3cf19c02bb09fbffca4684085a8a73fe35ff1bfe78e59bf35435c4051846f3f1f781c3e534b503f45ad0e406e9f5c3f69cd80c0d611733e080a49c016a88f3f49fecf3f463c6e3fc16c3dc05aa62bbf43c1bd3fa999373ff01f18c0c106183faa91a43f48697abf816c6fc0b45a873f09054940cdab593f717e0040f70892bf26542340115c7ebe303042c0da6fa03f46b73740c022dd3c51385340f8a7803f6cf853c0569f6bbf109bd8bf98b1963fe9bd1d4063328f3f68235e3e6a8ea9bfd82943c074a07e3fda0d3c40aef268bf391e7cbfb0ea8f3f2f16e83f328248bf447303c0c979c53fd12d42c028f9623fd01662c0d56117bf14265b408493913f0443ecbfe9978e3f329492bfe23325bfc0c29ebd7f99bebd0c8f8140c1ac90bf8c7080c09c5c93bf403114c0e598f23e050b60bfad06483ffade6340b386813f48390d40c30a5ebfd7771e40ac15efbeaed4dc3fed86ab3fbb406440293e70bfeb35f43f39a3a5bf89a65840c278ccbf8747624058ac603f9cce63c0d3127a3f3b35dfbf5a4722bffe5d32c07249573f66522fc0b64ba23f414ef43ff476603fbea55fc0f75f24bf9cac3d40721b62bff109573fe75121bf54ac3bc0a54965bf900642401c1b243fe3ae2ac05b0d87bf98d00ec09ee49cbf9a3e18c033c1a9bf9c9e5dc061c23e3dc77f343fdba1ee3d677b47c02accd4bef038fcbeb28eaf3f615c963f50788abf43c13440a93c823f8b987640bab4963f19f424c08f00983f7b5dddbf7d373bbf6a792b40c4b3b2bf98623fc08c9ba9bf28e254c0dcc21f3f31071940172d923ee0223f40179983bf6eca52c01741153ed8b921c01e0577bf8d8f3c40daa04a3f9dc86bc0e2068dbf81b3063db63d85bf70f2563f4c10953fc806f7be6563c6be69994c4040f38cbfc172d53f347b6b3f52b24640bc73183feb5b2740ebfe7dbfbc263bc0aea502bf9caa4440b9f6aabf51781fc03c63803fa748c9bf9d2fb8bf4a2e3940dbd7cfbeb1da3740485f6bbf72b45b402a32a03f2f8b23bff1320dbfe20a4ac09c55c5bf42241cc0c658f53e366f3140ffe6423f4bb5883fb397073e44b93c40d8c278be498f81bf02d6583ea637794032ccb0bf87be1040931d293fe74565406c759abfc6822f40218bb03fba184a406804a5bfc44e16c05c0771be68205a40e8b9493f8d603d4084ec9b3f7ac63cc0f3d9b93f6bf02bc0e562473fdda938c0d1362c3f41495a4099fcb63fee6118c0c27dacbd1036a13ff190a83faaca21be9646a83f89673dc0227dbdbfc2c35fc0e43740bf5bec983fb7a485bfd162c03ff99064bfd6bf45c0e30c93bf2568dbbf34c3b3bfb6fb614042a867bf4e126240d10b2fbff15711c05d6443bf065a81c051aaa9bf42b84d400b20613f5ac346bf6aa03e3fd4a75540b050abbf60a816c0c0e9b6bf91a859c063c286bfb11b4a40884a2f3efa6745c0c385ab3f53833540b717923fc5670e40cdc98a3f5467dbbf56dba53e5d632dc07113f03d3d983fbf508f8dbf5083493ead9c123fa2df66c0dc3db4bf85f3b43fc9f3983fad4e5640374308bf1b236cc0a05c293f5d37b53f5f9853bfc91235402873223f72669fbf1e22a13f96e85a403d952e3eafecc93fe0f5a3bf95ab77c025b86cbfb7f363c02aa6933f36191bbeb2e142bfe802e7bf007a013f95f3f4bfd0818dbff93b14c0e32266bf186a12c0a017b2bf778e6b4038e38a3fb22285c05b0e713f6eabc5bf4e78ddbf10a214400ec8933d636532c0ae719bbfb2db82bf4a1db5bf4fb90340db778fbf6f6136406614b8bf53e7f33f000f713e3cebb6bf51c4d23e081d54c0f7b3a23fa234d13f81c1983ff3292b407858313f5faabe3f96cb97bfa840be3f6d1f45bf735497bfef997d3fe04a0f40016d3bbf9ad35240b68d19bf06d9db3f3c85333f3d810fc0794e8fbf889d5540df6fafbf217ba23f681a4cbfc05e2240ebd2093fc1f63bc031802abf7cf6d5bfd367933fa7065bc04157ba3ff0753a40f329823f752b3bc012a7aabf3e6812408d1abbbf0a6d4140cc0ae53ec1f93840ca629f3e4bb35e4015f5a2bfebdca13fff4465bfcfeb04c07dcf87bf3e56cbbf042abdbf250b54c0a558823f717b2c409c64a33fc210b4bfeb599fbfbc1f73c03f234b3f6b9d41c06d2b643fbc276540c250823f6c5d59c0f1edc93f6f423dc0ebae383f2ecc01403a08683f834ff9bf4cd224bfad51a7bf1e83b43f707c0b400ab9033fdd8b2cc05b807abf5509483fc960a93f270a4840a16189bf53b6993e3a94df3e86d365c042c03e3fa927b13f706881be47713ebfd7c086bfc82a40c0ff9e943f5eda8ebfd8488dbf7ca45140af452b3f9bd947c0c1a8b0be112d35c0a79106bff200294069178a3f0eca3c4023ed063e58b60ec040053b3f741f68bcf7e9963f1e4f00404f7b2f3fd7f3184046806fbf37b85ec0d465b73ff73767c027768e3f42122c40c32e443fa86c00c08679323f9e8558c03e43a93f775338c07517a0bff4d138c05b9d1b3f25782f405f97883e61d3533edc7d83bf76dc29c0f18ca33f9f2330c01db4913f1afa8140362f053f735b0e400158963fd358f5bfc1f29fbf732da53fd200a83c412b1440e50f853e5bca00c0641637bfef1a44be27e2923f7fb29fbf086755bfd55b64c07a3485bfdbbdd0bf35452c3fb8d328409719033e48b809c098fb5f3e1a2b2ec01ee043bf897ab4bfc9fa4e3f8a7ca23fa6f011bea7e44ac0b8438bbf51e365c0edc0cb3db9a3fc3faa68863f41f7ae3f71d081bff86835c0fe05b03f1fd54b4029f1c7bf50e140c038f6f9bee0f709bf86584b3fa7e348c01c4ec5bf376e2640a2819dbf6f6d83c0224945bd45b0474002a45e3fd0f0c13f4377863f4ef62b404d28aabf651003405bd6e53e55821a40ae6eccbf928c4ec07e32033fb38f1c401d036abf65ced2bf059c9e3f3d6f1b4033a484bf8e8408c0bb60843f7fa03bc09f9c82bfd5eb46c0ff14fbbec8a478c0d46cb63fa0b96bc03cab6fbf3b201d4023dd803ff5f824c08499aabefd332fc0a2ce99bf8b4722c01640ab3f90580540ed2ca9bf80df35c02aaf943ff536e63f36c3b93fdee7cc3f2a0115bf94e72ac095cfa83fc4037bc0401d273f7ea412404a14863fef77234027cbc1bfc913dfbf95adbd3eafb009402fd3133f5ecb19c0cbd6fbbeae0d38c01b85bbbf89a758c085d344bfaa5c433faf9dbf3fb61f22c0330a9a3f337bc83f1ea730bf623018c05731aa3fb0ad3140c2c180bf109ee93f61d6743e533469407825aebf3a311cc0608c563f1de1b03f73dc4b3f594a94bf0f8480bf4a1f42c07e19cc3e236f943fafdf963f6a0226404907a4bf3c5340c005d7333e40a4234067bfc8bd0e6436403c04643f59b4013f41d5943f045008c0909ba7bf1aeb71c0e5cf58bf04603240dc0995bf8e0a39c0a6d99d3f19fc5040a73daebfdddc22c07ba36bbf045282c0cfe4633fcf5631404fc231bfbf041bc0e32ac1be1037a53fad26943ffbb14ec07bc290bfa92dcbbf6d539b3f478f2e40d0ff7cbf3d475f3f398316bff270afbff4e5af3f98f4514059dd5e3f2158d13f144280bf392a3340755aa8bff9fcb93f4e6ef13e2928a73fd52b8e3ed005d8bf3bb451bfca93a53f2de4b03fec0763c0dfab65bf5c5a5b3f39047bbfb5cb4440691429bf282e4540cc81263f8ee813c01469e83e9916d23f1b4bd7bfb2b15bc00fceac3fc0678ec04ba6833d68d93f40e3f7a13fb757c0bf98ef9f3f1d805340a190333fe724c13fd6c70bbf84ca4fc0c28aa73d3386483f2115393d954649c0a0a8c83fc4c3314038db8ebf1a8436c0b2d640bfbf6d4440c0ad9bbf8922bebf34e93bbf07592f40e5fec73f220f3e4015b1ab3f6b5721409f5c97bfe8530cc0c39f393dd035a6bb2c77663f99c75440708bb13f4a3b53c0191f153f8602f13f90af9f3e60904540b4259abfab84833f41c4893f3ead6e40508381bfd6c867c0560f943f89a606c0e5d790bf807b57402973aebfc1ff27c0a5f5b9bf7fa24a4066b8983f99d05b40a8a9a7bfad939d3f675db53fb1cbe7bf0c8cc2bfd02d3340984c913fe0bd56c08f980f3f9ad1d03e2118a7bfece428c04bfd473f660b34404a24acbf820fb1bf0c672a3ff6871bc0cabea03f7c1bba3f1c66913f6fb65c40d8022fbe6f39fcbf05319fbfb3831540c4d6a23f0bee35c0c801933f76792d3f7d018dbfb2385a40d0485cbf2df83640c5f49b3fb4ac17c03572943ffa3453c0209bbfbf10d85ac0435ed3bf7bda373f39c433bf648341c072ece23edb604b40a3e24f3f63572640df3a8c3fb196b4bf25f2973ff4844340d812debefc6f5440c3b8d1bfc57900c02299353f91ec58c08ef24abfbecd0a40e52110bfcaad2c40adb897bfab8a0ec020db9c3f20ee52408b4db0bf8f1375c0686d763f3df567c08bc6a63f7b051cc09088bdbf22da3640be688ebe0be603c008a8603f1e9f5440874f393f7c3d78c09a2398bf7e0f2dc01044d0bfc64e3cc069424a3f0d5845405481b33fe61999bf77bf28bf9d4d2dc0332b62be693e75c0f2ce8ebf94ec3bc0deb0003f1ef3f63f9ed888bf164b5b3f8ff0953f86cd0e40c60ccb3fdb59a53f970c4ebf45e7023e12a78fbf08f935c0a66890bfad71264048605bbf135260c033e475bf777e05c0813411bf02d75cc09c1288bfdd002240c9caaabdcf522dc0510293bf59b39bbff6dd203f17e21b407485a93f361e4abe47553f3fe62439c0215fb13edc8164c0934c763f4c3e3a40c87f9bbe820a75bf2499883ff97d4fc02c59813fac2728c0181f8bbfccd23ac0ad1aa9bfd292a23fdd3635bf2d2557403fee83be7bd61cc085b086bf3d9279c0b23079be46b4314080ce9b3ff5ef55c0e052523fc99825c0dd00b8bf315238c08c5b9d3fe4731bc0e87261bf5ac63ec0b1931ebffb721b40f353fb3e93c24940febe65bf6321e53fd41af8be972822c03aab883f7e6b38c0f92e0a3f9c266bc0c695af3f6c2fd7bf50d156bfec3b60401a60ad3f08b65fc09db3993f1747e83f0d6ef33ee98ee8bf500a9abf649052c0da8287bfa2e626c06ca1193f44af8140562d4d3feabb05c04a264d3f39a98dbfc3d3953f8c205ac0ad4583bfc1c143c0db9187bf618e5bc07d9558bf359e66c0013f45bf4b2e3940d7383b3f3be237bf4f0b87bf3b2166400f4683bfe2245c40cd9b683f5ab217c0f849273fb73de83ffaf8b93fc43db53f65f39fbf54a44fc08305dbbf9913e8bf9fa8cebfead14d408e380b3ff9c12b40e14964bf95071a40263c753f7a514c3fda23603f646c39c0af882f3fb03714c05aec5cbeb2af3c40b3f40ebffe1fcbbf254bc23ed1e503c02880abbfb2e434c07bfb3b3f841542c065254cbfdb262e40625a62bf07074ec0169ec43f2016c53f85202f3fc8b94bc0bcd599bf7cef51c09ce436bf4f52134084e5a63fbf2327404d04483f9f7924c0bbc8503f48843040e60102bf9f4707409f128cbfc7ab3240cbdc5d3e900439c0d2089f3f3a1c0440254da03f4a2440401023b0bf45b514bf5fd1663f1bed63409210bb3f5fdc943f1f23bd3e247b2a40927ba5bf724b24c017bf9fbfd1b71bc0db41e1be0a691a3f33a88d3fbbdf93bea02bbbbf3a6c53c06a5a243f06597fbf45ab65bf5eb24f400326b33fa63c263fcd6189bfc3e93a40fcc4e43ed40a86c0dae85f3f5bd97e4042b78d3fedb014c00f105f3ff4bfad3f39c9a2bf78972440ff3f88bf9089c6bf2c07cabe626581bfcce4b33f4a4e7640bda75fbffa9635bfa5a4e03e94b72c40f549f4be8d8b37c0155a2c3fe120744057e4c1bf06d254c05e184e3ee1805ac02bc49d3d9c560bc085f464bf2ee75840b8bd9f3f18cb4540da6a9bbf027939c0d7ad773f9f731b409b85333ff90125c03c91373fb61b45c00dbd2fbf21fdf73f298560bf2a77a4bfebc38dbfea4a7740968d763f82562ec0ea89943e65921cc089fc76be985037c047704b3f4f743cc0c1d46e3f4e9de8bf1b491a3fb9cc25c0309dd43f7b3e33c0a12b243ffa2d09400e4eb3bf89c75bc003de9d3fdbb2584060be513f50db56c03f5f18bf34635640dce78d3fcda59fbf1b2bcdbf3223304010636ebfee4470404300e43e51306fc0e1e4683fcf0e3b4069249dbfc8da10c06bb776bfd0b15340e902613fa40af9bfe69b37be90b1f9bf9969b2bd9d5e2e401aeb3bbf4bdd15c0e928653ffa19a4bf79dda8be3f0ae53fd2de9fbfc4272740914182bf9d64dd3f87d920bf83f45dc0cd947f3f1b9d85c0a78181bf7461e2bf13973c3ff4198fbf91f88fbe5236244056b743bf1e47fc3f9f63adbf23c68140b383b2bfadada5bf96dda83fac7d60c06d18663fa0cc1640dc3d6a3fa94ea93facdd1bbf0d536840dd0346bf5b8a4d4072c84bbfe37e33402ee4a8bedbea33bf980d4ebf088f583ef9a3b03f5afd17c02c2e79bf952cd53aaf90553f357c47c008b8983f1eb306c011efbf3fdb5484bf88f52ebf6e32debf2f1393bf256c2a40a92d773fc99d823fb05064bfac7a3c40dcbf753fe0313640d185e2be142d4cbfd836bdbf21640640a41e87bf8f40f63f6a776e3f2f444ec0dce399bf43935440868c5a3fd9a7f1bfc501aebf00e124406eadb93f672cb03eb1f603bfb99c6cbf8b159f3fc49b70c04bf5febeae9829404ec492bfcc9834401929843f6161cdbf4f248f3f3e6539402be8913fc8b87cc0cfe26bbfe469b53fc0fca3bfe1d429403f05fbbe1ab2eabe632198bf880b67406722ccbea95f344047225fbf10d9504044ef723f319432c0b732583fc7e13fc039c2563f3db33c40816a92bf48e654408f92a0bf4ac806c0a59eb23fe8df67c0e83f14bf270f35404064493ea3391840e8949e3f3dc22cc0fd60ae3f41a6a73f3b9e5d3f9a6051405c0cb3bf735d43400714a33f4f6b23c0970e98bf3fb8b8bfedbca4bfb9e4f63f9870543f483ecb3fd632933f587d1c4034c638bfc1b23740fe071cbfd46552c0f71d963f72da3ec0a0ccbe3fc7312b40708ad9be370e2040cdf6b63f07c606c0045a71bfb8065b40ed4fd23eeadfe73fef00763fe73dcfbf9d83ad3fd3eb76bff697123ff5a8a0bf5acd83be9d7a0a40d970a43f9ebbf53ff92f693f94ce3940ae458ebf8cc572bfa4cd9bbf227023400435943f95281ac0b5a3a9bf99f5b8bfb26610bfeb032740e43b5e3ed6316a3f8c3dbbbecfb16140ed17483fdd6c82c047f4613fdbc060c06eb40e3fb5d525c0f35bbabff73b0040163b1cbfcd4543c0de4a0bbf75545340b30baebf2107a7bfbc119d3fb41e4ac04412bbbf52f034c0ea28bcbf53e816c08b682bbf4a9c7f40500850bf3f8f0fc0cec68f3f1a634a40ba518bbef70874c02e8cfebe96a42840abd14bbf816349400a74b13f07bc6940830e9d3eaa9b01c0bd6b713f007e3bc0fad9b8be5a0525c0731f9a3f7a19083ec3d3253fdbf009c0b674013f41600840f901873f0d23833fa67b03bed2e61bc0971fb6bf42582440e00aa03f068848bec8ac71bf97b725bf0dead43f318a18c0bb6d95bf930a0640d5ea1b3f7bc960407698c8bfb36c46c0a0fe68bff41b51c05968cd3feae838409de3a83f1906db3f9da00e3e4fd855c02affafbfed4412c001c472bfd5c7224037861fbf92f3acbead95483faa58b53f3e9f033fd5086ec065b0a4be3f8f833fe86698bfc7a706c05e678a3f6a800f40422955bf4df152c0ba2f9abfeff86140f744abbf7216e43fc172b63f37e01a40b9ac96bf0437084045ba443f06332b40632cafbfb7d0cabf6b0fd1be58fc983ffb3e3f3f54ab25c03d46a03f85e15040570ebb3fd0eae53fb9330abfe73b14bf4fe2b9bed02570c039e73b3f052034405f6e93bfed2702403c01b83f5052fdbf5dd310bf38882bc0d27a83bfd93b3040855ec7bf74d043c02782ac3f0b331a4033739dbf8715314046f63fbfc56e27409be29c3f6cb0424066698bbf2c3483407c46aabfb5257e3f8e25753fdb3244c0b339173fac1822c0dd18923f5f4b1a40fd869fbe76eb4640e967afbf497c003f8eefae3fe1e904c05e27a73f464144c024606d3fbf9b0240eee0afbfce98e7bfff35933dfd975a407c1e9d3faab8e43fa799803f0a17d53fcd07653fcb16cabf9982da3e044537407fd21c3f2bdd5ec00a5fbb3e0955893fa518b83e7b394cc02d77153fbbf03f408f7765bf8cf7a6bf3752b2bf068931c0f5be953f3ca25bc0cbabe8be15ad7ac09547923ffa7609c012f2b6bf24ba27c047d3bdbfbf724ec0463eef3ea75b4c40cec050bfbe116640e546ec3e668e3740ea4b45bfaa593440b1d31abff76337400d747f3fb58d3ac0089d973f8272454028c6a8bed2abc8bf624facbfe0a0efbf709ea53ffcc11640c01c8c3f366e4cc06e28a9bf5512d43f84bfd4bfa4aa25c0fc27babf65203b403c5f9abf3ac8fcbfde73453f4de01340fb82ad3f908ff03fe5b579bfbf04404028d3703f269136c00ed64cbf809056c0705053bfaee7d33f4be57f3ff96e6dc02ad2983feb150c40e971a2bf8be516bf6dc89fbf40ec50c09bea733f96a2b3bff361b53f16fa26409e2963be66a5404015fa7b3fdd840b40b458933fc8a45340726db9bf69561d40aeca703fbfac2840ca9a823f0ed946c0ebef4cbfb777963fcecbd63f329383bf0b3d5dbf7ff21140c1f4993fb7e92e40b6cfa13f6da94840e1298b3fb8d86d3fdfcec73fd5e2573e5b3f53bf3078ee3ffa1489bf943921c0a4e122bf52b128c0907ff23e15611440405b9dbe82080040dc002d3f591302404c4305bfbf4317c0dfb9933f'}, 'ds19': {'d_in': 8, 'norm': 'minmax', 'nprm': [[-1.424435799613096, -0.4926082740914547, -0.5190096506165479, -0.595131581104708, -1.0229620310581755, -1.2025874100457732, -1.682580073685095, -1.752365552582784], [0.39044519870842814, 0.6855956883458527, 0.8739954148897571, 0.8899185650628819, 0.43833388982230287, 0.6080178311404043, 0.3781401530129683, 0.5262506764092122]], 'pmean': [2.926431095938758, 1.1590924006845749, 0.5900186498451228, 0.6671156733443804, 2.0600762039522102, 1.3816488461148955, 3.29294942964362, 1.9757113871057315], 'pcomp': [[0.643713692174225, -0.09461093844948865, -0.0006096324306390073, -0.0021294513554760354, 0.5308595903412768, -0.18207209158929036, 0.41774109298594025, 0.29530793192236443], [-0.27269830881617196, -0.2811475279115552, -0.0052965174408777585, 0.003316767402548498, -0.3090177477411208, 0.3221717427105495, 0.7927670559090859, 0.137063637483142]], 'd_pca': 2, 'n_tr': 912, 'ztr': '3f0512c03cab6c3e596f8dbf5699183e634b71bf5a53903f251f9e40971fa9bcfee9eb3fd6bf8fbf9e9400c061933a3fb185cdbf2049b93d3f6990bfe9ce14bdf24f08c07255073fc10410c0f4a3603e5fbb06c08be3c63d15688240354998bf02e520c094f2debec54f6ebd5f18b23f3d620fc0f1339f3e3e698040347550bf0f65a5408ce33dbe8cd0e93e109b733ffb63143fd595a3be6622e4bffed0913e454546c03ee22cbf181b3d40004b93bfee029740d1e24ebf04d71d3f2730bbbe5d1d47c0f33c92bf35c7e83f94863f3ebf6c51401df6a2bf550a08c03c06003eb6a11140ebb1b5bfc788d03fdcc2a83e52ef05c018b51c3fd240a13f8e6b4f3f8db084c0c31da0bf2c0a42bf340f8a3f75d502c090500d3f0e089f404c2c973e48629bbe1c4249bee9806a3e5cdf4f3f7b7c71c0e28fa4bf08e52540a48b40bf75989d3f52b509bf812ca0bfe5a4983f5f06953d4912c83fc4979f403ffd38bf14fa12c06fb0afbec14419c0e73bb03e7e9b9a3f9d9cf83e2cd4a63f52841c3fcf0303c0382a153fe78087407a453cbff1b135c04b2701bf1efa86c0bb8496bff900883fca7c453f78cd4540a53656bfb6172fbee626c53f86ddecbf7ba94d3e9c8957405e3f7fbe9782dbbf5a2c9d3e4f1e2940f631dd3e073b304034c186bf493e473f9d6f62be7dcecdbe6b7bae3fa863d6bfd2d8a63ead1394bf22fa653f0cafadbf7154a7bdf1f7c9bffce67d3fca2511c06103bcbc13f4173f314246bf18de00c0e831d8bd8f7bd4bf81b25c3f4a781d40614c203c896d38c0174b9e3dbf64f5bf3b89203f7f5cc43ef6cf25bf442c703f252f393f7bba083fdd77ff3d0435be3fc9aa2abf44c87140b35e8bbf636c363ff159873f1a37a140d0cc993f37f9d73f4d5efebe440c98bc2867d1bef98f983f0aa0173fd7f96840572ec2bf8ae3ca3f987b283f0fff3d40c0736abf61a2b9bfb47e6ebd399b7a4075e288bff11886c0bda681bf999cd13d403c05bfcbee66c04c0fafbf9c4474c07d46a5bfed1aa1bef4ff9a3f45159e4040732e3ff2c3b6bf2949bbbe28ccd7bf7c043dbfa8b7d1bf6d4f2fbe1adf8fbe38dcbe3fb93ebc3f4a617cbf90cfd63f88e7043ffb17c6bfd8b3593f01ee5ec0f7786bbf3c9b58be563e7fbe2b2401c0b75c2c3fae1ae93e47c356bfcb2424407db380bf87a54440454f6fbf1f539a3f8b1f8dbf765e41c00c9249bfcc7011c0a96ce73e9504a93fa608483f47f27b3e5051b63e416e8b401ea181be098c443e37c5c03f1b62b5bf55d6933f5327584094c865bf1b1e60c0b0c8a1bf436355bff9657c3f8b7510c06280e53e429951c04b568bbf0cdf90405ad890bf0494263f0928c03ff62a8b3f9964b7be929206408e562a3f4a0d5ec0802172bffb1f9fbf6158993d2ed469c0cc6b9ebf8f8e9c40fd0d93be13e146c00bb2c8bec413f4bfa2d67f3ea0969a40fd805a3e5b8cd13f13fe24bff50105c034daa13e2b007abe985ce4beab3147c05e3e48bf17513ec0ae5742bfe81bdbbf7c1dd53e72d9213fdadf673f28bc3ec041733dbfc2824640b467f93e45231ebfc9cc9e3f08a95640ed10acbe5bef4b4048b655bfcc9b77405b596cbf49f0d73d3aabbc3fffb6bcbfb01db03f7a2c9b400787a73e52044fc0ff4bb3be0cad10bf3984833fde9680c0dd55cbbf1ee6d6bfb0b3aa3f891491c0b137b7bfe53274c09eaa8dbfc88f1a40ae0546be10e641c0ccfd49be8a7f39c0411006bf2e900740de3788bf32c81ac0d0cf04bf9e67d5bf334c083fd1c8eabe877047bcd015933f915619bf5d093940355992bffbe1b1bf3db29f3e7ed963c0afe7a3bf466355408eab89bfdb61ebbf9171063f3d489040ade683bf3612ff3e64fd963fb6ae923f81a77e3b57fa75bf7ea0ba3dc37d40c0fa9e36bfff7ddcbf8598c43d9cd47d40e1074cbffcee20c093acfabe8ea22fbe0205c63f678fa1402af825bf72edbcbe4d931d3b42e6a440e364523e1f2690403f32f0be1c7858c085cc26bfaaf23bc0e298f7bedc839640202429bf943cf6bf6ac92d3e830e36bf28a5933f50ba7740b7cc80bfc324f6bf0af0d23e77ce09c038a8423f8afc97bfde0e593e624281be9007c23fe63cb0bf6fd0683f8eb6b0bfdd5f853fd2cc7ec0346ea2bfe3f35a40f6399cbe080898bfc28da83f88369940d5d709bfc21647bf5b2dc53f1c22c4bff2a4b83e2a4f9f3f4ce4593e5e7c0a3f5c46dabe675ae5bfb9a0213fc82b533fb7ae953f2189a0407bcb103d2ac604c09517a43bd24112bfce0c8b3f09b29d3f4e156c3e7f0e36bf2439e1bde94bc83f58ba633e65b13940b98e92bfd543633f914c2b3f072566c0d0cc99bf9f7fa4bedcfb253ea479984025822d3e2d96a83f7c0606bf18e0fabfcdca263f58018a3fd34e423f9304c8beb0f536bdcbcfa73fa2190d3fd3bf383dec8292bec5088040b6f375bf55436c40289c19bf85309540f6531f3edd30f23fda6af9be3ce37fc001b4c5bfbe56f5bf47f5253fab47d5bf0a22233fa3c03dbf02ea8f3fd8cdd5bfbda2963e16a49d403e73e63dccfb0040a7eb82bfdc643e3f8e13623f7d50a33f4272ec3ef72fe6bf649cc63ee08f19c02cfc91bec20248c0b08470bfc9b10340237275bfed0c8cc06305b9bfcf7e31c0aadd46bf7e67e4bf3b58233fc4cb4bc0f141f5bef3f18ac0dc1fc5bfdd1f08bff150063f5776624001598bbe4a02febf61f51a3ff45595409503743d84e1e6bfc503113fa5287bc0c60790bf212f9040058d9fbe3f8fda3e66d3bebe0a2804c0343f093f511c57c0305114bf79f407c0c47a5f3f6572bd3f468704bf118a1a40159ea3bf109079c032d9a1bf842803c031de123fce7e95bf3010613e3247d2bf76cffc3e6ca190c05bfccdbf3804aebfeb274c3e6cee3c40d42d893e55c08d3fdfae983f26440dc01803733d7a9b9740753db1be68f6f6bf777e153f167cb93f4d4703bc1597a5407df31a3f9feb8ec0ca6f3ebfc9ee903fbb20583f9b94fa3f9c34963e32fb73bfa99b3a3db4f4b23faccd07bfdebd3fbfea843cbe5eb9da3f139d20bfd33e5f40896796bf35ae92408878abbd5f4c34bf03c2293fbc6d41be318cc13f738c9f3f1f44063feb94b43f56e03bbf7bcd7d4066f196bfd11da63f265c0a3ec749124064a552bfc1ea79bf17306d3fecd7f3bf1585943eda66473fea6ec2be875ca040833ba83e7ca78840eb2e8cbfd2a59e4026958abe2902fdbe0c21bc3f2af7783fd77b543f8b7d893fc977293fefab8140c040f4be0ec2ed3fc2571fbdb0b756bf30118e3dc5fd6140b50467bdd8224fc0deb82abffbdc29bf84549f3fcd008a40d70343bfb93127be83f0d33f75b64dc0912a1fbfeb511cc099851bbefeb64ebf06869e3f5a22ba3fe4b9ba3e999d0bc0ac8e933d9afef33f7c1d80bfc33f533f8b98b5bec3af41c04b7c35bf5dc979bff0c98c3ef952f7bffdc40d3f7bbe7ac04d0587bf2382ae3f465149bffcb6a73f6538423f9f577840e6659cbf8c62433fc87c763f222caf3f852e043f20a5e23f36a4d83e32c081c01d13a1bfe35a3ec0cb13ccbe6f76a340d8fce5be8d24923f9572e9be6b4cd7bf73de1d3f37f2383e3902d53f439cfebfde4d893e9ed61dc0741141bd55c71d4023c017bfad5612c03118313f0ad3ef3e5889c1be947b3ac03c0a09bf5d69843d4a84ccbe2c210bc094d992bd26116ec0ecdd86bf45ef57c0d60219bf46c89e4037b2bbbe52817f406818a7bf4b898dbfcb08813f4c909a402723d53e3964273dd5abc93fa3ce9b3f02b3893fb23139c09bcd9cbe8802b03f4b3c67bf0e3ae8bf07d3eb3e2b25a83f2608063fd0defabf7ee18b3ede917abf91d395be8f5aa8be17209a3f54229d40f17f95bffce59b404ce6e43facec253f265623be599fac3f2370383f713412c0585dfe3e00a18b3fd422b3be99001ac082f9cabed624033f8e91a43f26b051c00d6d73bf06b19d401bba95bff255e1bf14f945bf4015f7bfbbfd103f0c2190be198eb23f85a1be3fac912f3ea4ae5ebf62725fbd19991b40a8b59cbfc3c1b9bff56ac43ec9d400bf4ae6983fa9198640681b55bf3e330cc0d04a683e907a2c3fc7d8ea3e518cb4be50d7c13c7e1495405199f43ea81e0cbef231e63f91be34be2306e43f4bd09240785271bf2a771cc0d648b3bef8d2a440474c173fe2c6793fbdcd933f76030c3f35453e3e8425453fc876103fe87d774037f89abf855c413fe412913fd38fb83ff7ca74be80bd33c0d6336abe34c90dc05de8afbe779b26bf8022c73fc7f58b3efe33a43fc975d5bff017b83ee70a52bf6e0b983ff49d5bc019b464bf6c02ccbfb82f7f3e6305693e49e4ba3f97dcc1bf648f743fef1ff33eb829aa3f857332405bcb89bf04e8f6bff55af33e62188340b3fa38bf9f2f56be7233a13f841bc0bf2bb80c3e5b41e83fc4f0743e401961c0abfb3dbf304011c0494a963e25bf12401f02033f49c69b407570a23e6deb81bf2c01ac3f588685c03f10b3bfcb60673fce18813ef7d0b4bf00b30e3fe1b211c030e519bd4107cb3e6e229bbe3bdc1a40941a613fb6b59840f82e603e127afcbf603abe3e219ff5bf4b063b3f7b36eabf213c133fe77ec03f9cd5a33e205bab3f69111cbffd185b402cfba7bf217941c0a882ffbeeec5b93e72a6503f7069153fe792babef39b88c054c98bbfd4ccc1bf301e833f317be0bfd65dbc3e8f636040648bb2bfcd568b3f3a33813f1b6369c0a09396bf5c639840f17038bf6f5c92bf6776b93fc80c8e3f567c803e7c10e33f31a5f2bea463a9bff71037bd758712c08411c03e98e898408b7b1d3ec4e383c09788b5bf1dea8fbfc332af3f8a115bc0bcf35fbf554c67be528706be55fb843f20ae1a3f83d8a5404f8f68bd9bda6bc07d90babfa5566fc0177e8ebf7b6158c003699bbfd4471c3ff4dfab3fb1c68f3f70d2583fa1521e40dd3613bf30d28cc035c1a9bf36e620c0d33e3d3e74098c3f7958ce3e0eadcebf449a693f8c8d6d40d36d4dbff7d679c0256d69bf5364babf374f223fe9eead3f0fa50f3e0c535a403e70d8bf9585973fc79b693fa17043c04c1d02bf77158b402646193f6773f2bf7155bc3ec49c8fbe572584be3e521fc0de2eb6bed7f76bc0cd52d3becf70263f60139f3f2fdecb3fc3050cbf7bffd83f3b164c3eb286a8bf023c9f3e54ac883fa7e3723f002f8bc0898badbf6b0e88bfec8804bee5200cc078f1b93e7723c53ed3e8e53f0781773f71208f3fad45b63f6def123f50f19a3faaff663f67518d4051b210bfae5d38c0adf7f1bead98b2bf5968913fdd8ca240c93911bf234d07c0e8700c3f033ae73fe991af3d8907aebf84609e3f7640d63e146eaabe9596d03f46a356bf8ab7f23fb89360bf24e80c400c1433be75505fbf7379d63dc8889a40fb8f623d40e504c033d1653dc642debf341d4c3e08a69e40fab2f53ee9ffdcbfe5e4a13e4032c9bdfb6ea63fb2c132c0a6d299bf58748abf0196b73f2b599b40ae06e23e0e88483fa87b4fbdb46a72bc2279423eeac4b73fcb27fa3e09bf0bbe08c5ca3feb881c40eb129bbf738455c0826c84bfda6851c03d157cbf5ffb773e2b68b03f3fbbcabfa3ba8e3ea65f9d4018cedebc9b340fc041d707be88f8c43f946e18bff925a040ad10653f44cefebfc504b23e35d0f63f59a56c3e4ffa63bf4398d33fdb7b6bbfd1b42d3de703c73f18ba10bf6e8c1ec05b38de3cfd7cbabdac7072beb9f43dc010f039bfb806b5bee8ff993f3ec49d406a64d3befe28833e1e99a7bd4645123e13679e3feb19e83efd0c14bef7d18d4076306ebe8cc08640c3f663bf2e7a02bee44ab13f51bfd4bf682acc3eccd6c3bfdcf49d3e2fec1cc00d7608bf074076bfbfc4803cc55d31bf291a8abdb54e8fbf19ff563fbfa61640365c16bf1974efbfe39df6bd2969c03f241bbd3d054601bfaa219e3f6c9bf7bf1531e93e7f2cddbf86a8b13f3660da3ea334dabe46fa9f4057420b3f1d2e88c0680a36bff9b02ebf25388dbe6cdc29c0c7c265bfbcc28dc0ab4798bf1a929640d9d009bf51559c40425d40bef7f9873ffe3b56bf743416be24009a3f6c4b80c091639cbf7534a93f6d7c80bec932aabf9f6f5c3ee77a90bf7a7f663fb1aef3bef4de903d79c888c04944a8bffa738dbf1697d43e2665debfa897773f63690a406f1d81bffb5a82409d31843d4d03044034df85bfabb38d3f70af20bf01628ac08b0493bf3f64454003bac4bfc5c523bfd4426e3ff14f9240808a23be5f460fc02da7c63e3bf950bf51109b3f31466cc0b71390bf2d7b03c09566b23e439d0ac05b49ad3e70f02d3fcc3244bfdd8db2bfa6857e3e677138c055c6303e3204a0bf0afd2f3d5c6e2bc0dee3e1be9289a53fdba5463f0b0fe73e9455adbe8389173fa872ce3f7157cfbea156873ee02c4b3f3c98aabe1cf8803f29be8d3fde32964006acecbeb78d913ffa43413fffca19406b7aa9bfcfd59f3d943ea7befea7c0bf0a6aee3e03731bbf9ab2a73d98de06c0f9ff6b3ea2608340a1934bbf7bc575bf7479403e488bbabf83d1ef3d363c60c0842d74bf6abd2dc0857720bfc8919f40d80b1ebf0491b13f78eb383e4d24923ef0fa8fbea16906c0ded59a3e5b9b1c408c25a6bfe59d50bfbc70233f579a24c04d7a01bfb2c494c02cfdd3bfcdf5dd3ffb98803e80d47f40cca84bbf1f601cc05350af3cd493583fe7cf6b3f85a2bdbf00362e3fd282fdbf4e52573efd5aa9bfbc08703efbbe71bf61c2d43fbdc829c0ff4751bf690c1abef691023ff3ff0140cbb2a8bf20ae5e40740f91bfd9088bc0a2a47dbf1c26b93fc1b5373f4bb59ebfa591833f39677b40f99fe6beb3843040ddef87bf8e10a0406179e33bf8037abf3a885c3fa7340b4037c19cbf65aa13bfcd85a3be5b195d3f00c226bf85f44ec03ed180bf5b8424c01c03fbbeeb67b83f324d62bffe36ebbfa186343f7b91edbf7a98383f8ed2af3fb13ec73eceee2dc01bab24bf070ad8bf7f48863fa27c243eae43963f269c98407168c6becacb3a405ea91abd68e460c0399e8ebfff20563f4753423fe85ab53fa0c6e03e952592408d414abf0ae435bfd059ddbd24e755bfc878a0be120aaabf49dd0f3f78f5603f98d07f3f996ff6bf80ccef3e64586dbfb8314cbd52c3b4bfa764793ea9dc24c0e511d8bd2d4d34bf6eb9953fcc7f03c05945e53d726616c0f992393fca10a3be1a08c6bd82f6e03f2b0538bfed690fc01288803ea764a5bf77c579bc0f4f9cbfead74e3e280052c01a8a46bfa99bc5bf31bf7a3fe7ac09c0fb72e33d121a4dc07fd280bfd33a49c01576aabffc1690c01eecb1bf1edb3cbf2a4949bd53e777c0f3da76bffb7d8440ee7e59bfa1f1c5bfd9497c3f351773c0730585bf81509e409e9bb8bedcd1aabfe4559a3fcc2944c0355a9dbe5173693fbb61b63f8a19b2befd3069bdc3185ac0d7f9a8bf0ff378c059af4dbf999708c065ffe53e31e200c06dbeaf3e9974313f96c8dbbe501eb73f6628ee3eb8e09a3fda31f53e77542ebf98caa23f187e2a3f52776f3f3652964041b43ebf0affefbf8f56a13e59ea18c0764fb13e2b408540a919813e68c08440307b40bf88eb863f8e48623f65a6a34034f2543f7f4e9a40a0f60cbf82870cbf3b19d13fea399140e9ae793e25411ac0f12ae5ba3da5914034de0cbe15504bbf002c393ef82e98402db856bfd7ab4f40a9e560bf431e0ebf2592b93f1e78e03f89f146bfa9d4fcbf33b4a83e9775b83f0041d43da5841dc0b92e8a3c95cfb1bed8178c3f26ec22bf1f9d143e63dc3d3f6a13e9bea2f3e0be540eb93ff8a241c09ff031bfad35aa3f4e06823e8d3ca93e23bf00bf838a0ac0a87d8abd36d024c0679e3cbf20b121c0796e74bea228b63f42892f3f5495f93e7b2f823fe210d8bfb125be3e7d7085c06dccc2bff83ec63dd44fb53f5c4d0bc0b716153f741e17c0d498b1be63ada8bf4bda183f5b220740adcb98bea5b210405e3090bf99867f3fb99c1b3f14f083c043db96bf812b42c0439801bfcb04833ea516b03fa6a464c0a9732ebfb596b83e73fa65becb0dfbbf8b50c83e0465b14062d6cd3e999b86409e969dbf72ae9440f6615bbd9b4e16bf6db8be3f39cd02c01373d33e1cb8873d6c77a93fb40b4d3fa5a000bfd0179c3fe3eb233f3d387fc0d9d163bff64b9f40e7f5763f5e97f03eb6394e3ecc7363c0d1fd88bffece0740812766bfaff3a3bf15e15bbde89826c04ee1c7be130d89c094be81bf411646bf831571be20812340b36907bf49e46cbf5507d93f8fcb9240baede2be8b2521bf2fa09d3f7d3332bea6f95cbd639706c088f7ef3ef0a65e3ed5d573be91f144bfff597f3e1c9f7a3e0954bebd28114fbf96ec843c8cc9c6bf82661abfa635f9bf18b4523e051ea94061a23b3e304b103e14bd9f3f60c0bcbfdc2fa43fb62187c0afd1bcbf4971603f2ad821bf53976a4014d35a3ff18874c05e9f85bf606386401d3577bfb528a040dc53693ed488c0bf69b5fd3d228ca8bffb5cbf3c7d0d94c0aa2153bfc92fd13f89176dbf3b13ff3d9c0ea83fcd403f40b77ea3bf0f14574094b0b1bf93f5b2bfaceb853f6e1af5bffe19583ed4626d40048eb1bef65f2f402d3a6cbfb80cf7bf9311703fa57631c09de812bf9d3dc8bf976d933fadc786bf0514243eb6b3a7be1a9f98beca8e9a406f4897bda5fdc13ff0aad2beef86c2bf2b5b7a3fa7fd9640258f14bf8a9288bf9be9833f2088d53ea6e4a4be55b025bfbda1bb3ff481bfbf12bc633f9bc105c0434c96be1538fb3ff4c937bfdf75003e3e766bbe682b2340879d73bfb484884008f38dbffd52c3bfeae4023e2a5503c0e35f643eaa8131bf9a6b0cbe5807cabfbb99b13e04c8fdbf2878df3eb053dbbf6858da3dc12e44c0778432bf83ea01bf5473bd3f728faa3f635ec2bd965791be071cee3fa27c1cc0fdd89c3d626a67bec0d7ce3f236562c0466bc2bf1dde8540a971b7be02ea83c0ae1c99bf7eaba7bf3335023f5730384083cc6dbff7b4943e2a9d22bf79e905c0ec157b3ed7fc3bbe6301ca3f09530ec0aa02143fefc649c06cf089bfc6c9a340d5642a3fa4b958c0a17c56bf68c29bc0c8e8bcbfcbcc593e9e1bcebc315d03c08139c13ee9f82ac010e6fd3bbd31dcbf2436023c760675c0af0794bf5f2219c0ad5193be7d8b7440921099bf481512c0bf242cbc0e8bf93eb5a36f3f862580c098d29cbff3efa2403f54e03ef2325bc003665abf1a799540ef4587be6a8451403b198abf980328c0546e36bf459d88c046c3b9bf00e99340ea6c87bf1aa85b3e46cf973faefdb63f92ca203f92e68540b3e289bf9f3d04c097f83f3e7cf151bff64520bdb6c25040f0746dbf5466823f734c663f5fc01f3f523c17bf33c143c025031abff8deea3fda6c713e68effdbfbff2a73e858ac8bf1cdcf93d471907c08241043fad7637402364adbfb307824081e36dbf4ceef93fad9c2fbf444b1040b2ec86bf3533a340eff9e53e56441bbf389cd4beb9ea803ffa6ca93f1e2f56c0493d23bf05d19fbf4027933ff6ff54bce6a1a13f1e6e3cc00ca852bf80f6df3f61d0693d7421ab404d60113f8f740e3f45e05b3f4e0903bf56efbb3ffa42e6bf107f2a3fe832eabf84991f3fe33304c027d6ce3dd9f650be078dfe3d180e33c03c1d24bf3a216c3f6ea71b3ed1a869c0931088bf0a8f8c4072053fbea5c24cbfe19670bea20a0f3fd169c83ee4ddb0bf65f5adbe18d0c93fcd12663e940cd6bd24a03cbe1fbd00c081ebacbe23ff4ec0c85152bfa57c9740be995cbe6eb07cbf4149b63fb9503fc0654978bf6af0a5be7f7635be030e13403d2786bf48fd6ec0056d69bf3ff186bf0ac0453f2b1c07c06dba013fd88aaebe43e407beea6d3c4053b229bfed0f1fbfa487503d9806323f9bfc3d3ff1b12bc07bdb0bbe1ee63a3fc2281bbecfa1b3bf50368b3d6f1505bf190f893f6aca613f581f3ebe22e29b407d882e3f908589bf65c0363fac68d53f1aaaa9bf8d8229bf84bdb23f3962e0bfe9e1ac3d5ff206c0bd511b3f20c758bfba4d053e01da5dbe539c0cbf287f12409d6f8fbfcf0984c0654796bf3b749ebf5ba27f3f31d212c07e0bdf3eec0ad5bf343dc93e95bf9d40fe9f793e4e0849c04f8732bf2c3b8b409ff406bf4e0df23fa39165bff867c53f01fe0e3f047b47c0ecc8f8be28298d4051a61abf198bdfbf58a25c3f3bee5840b34799bf3fe3c83e440c883fc9abb3bd1a858c3f'}, 'ds20': {'d_in': 9, 'norm': 'minmax', 'nprm': [[-5.062548102455182, -6.154774827195349, -9.857384732120424, -9.903987101830234, -5.458709533658219, -6.71047785162911, -6.989720996765667, -11.794920162668872, -10.255193605291092], [0.06544417001829367, 0.11561828544274379, 0.05004809258976183, 0.04660677340827113, 0.05743290291736619, 0.0847785688249009, 0.06595689906581656, 0.04561161333050732, 0.06175548979330296]], 'pmean': [109.82957410792781, 34.12495348985671, 206.88072884827918, 216.12118961829958, 123.37186116146073, 65.72208857043228, 103.37118426412239, 260.67585062680143, 153.99005881693046], 'pcomp': [[0.31761306424577807, 0.024315853427299028, 0.6109017970200469, 0.0022652993972429314, -0.48722446190316343, -0.12053185457054431, -0.3309720135377963, 0.03935116097156109, 0.40288702095092765], [-0.016620640206697377, 0.0011096191728070066, -0.025873845458447887, 0.2671402532054683, 0.01374739302623281, 0.005062055018571437, 0.014657861777074534, 0.9628541668756824, -0.013097232728815333], [0.026822320706316655, -0.016950224593404666, 0.011502216767964858, 0.9627817053402752, 0.00879384732472183, -0.02595303507743671, 0.005191798098386576, -0.2665300914715052, -0.009808346540106198]], 'd_pca': 3, 'n_tr': 960, 'ztr': '...'}, 'ds21': {'d_in': 9, 'norm': 'log-sym', 'nprm': [[0.0003648937942922559, 0.0603822478564162, -0.0006340610019192374, -0.011709704337167025, 0.04311890324613271, 0.01086750099716826, -0.0034400254828874143, 0.02544085852773749, 0.011798595436462873], [0.6063676239327099, 0.8319519431303453, 0.9357622097465329, 0.8733426155004633, 0.6957268362361765, 0.6074917757243907, 1.021618628881539, 0.8505951395547321, 0.9904136052623965]], 'pmean': [-1.897817136111379e-18, -7.591268544445516e-18, 3.416070845000482e-17, -2.941616560972637e-17, -1.897817136111379e-18, -3.795634272222758e-18, 3.416070845000482e-17, 1.1386902816668272e-17, 3.6058525586116197e-17], 'pcomp': [[-0.04603297854930887, -0.1452304299739383, 0.2943705636128212, -0.41775217298515543, -0.34845674030590246, -0.021593691958080984, -0.33741562058317687, 0.4715392091390249, 0.5074753847960818], [0.0003875114463330202, 0.5751389375810219, 0.507930852826476, 0.3698411039436672, -0.19544037245923646, -0.05089293417932723, -0.44440167875145836, -0.14635644369151357, -0.12140332890914418], [-0.6987795829899206, 0.00697256094465909, 0.02463313621697496, 0.015530982945940507, -0.035283849847374975, 0.7125663629498712, 0.0014207458101925846, -0.03646889377726077, -0.021970538443565894], [0.7125370782278941, -0.01749445236157976, 0.03905502090408761, -2.7136787481243146e-05, -0.1099053990489431, 0.6913562389192556, -0.003665923285198157, -0.018898925912491604, 0.0060255013015819], [0.04100675846952416, 0.2833522562241058, -0.11158557458107285, -0.015259496395363786, 0.7902221519331926, 0.1056659570953474, -0.297221029555854, 0.4134280633599138, 0.1023049563272985]], 'd_pca': 5, 'n_tr': 936, 'ztr': '...'}, 'ds22': {'d_in': 9, 'norm': 'log-sym', 'nprm': [[-0.014816116453691889, -0.2571462700732452, -0.17900808097754473, 0.412321680338496, 0.01905253600172145, -0.21202240620113502, 0.3762569656130009, -0.15919235970375645, -0.09404991117980237], [0.5101420108112427, 0.8998304284814278, 0.6552052611251284, 1.0951363782879782, 0.4841905533635821, 0.8360361582597027, 0.854133367698037, 0.6262076774557366, 0.6486522079974943]], 'pmean': [-9.251858538542971e-19, 1.8503717077085942e-18, -2.2204460492503132e-17, -1.4802973661668754e-17, 1.1102230246251566e-17, 2.960594732333751e-17, 0.0, 2.960594732333751e-17, -3.3306690738754695e-17], 'pcomp': [[0.03576213651079927, 0.4323383755702288, 0.3787626321004609, -0.4155636776517163, 0.010042720365629733, 0.31905454788768045, -0.3710067208924686, 0.3312717896768182, 0.3825789468843926], [0.005169353826814551, 0.010038089532649955, -0.39612850903068864, -0.2713115494188033, 0.14718709378399308, 0.5423159789890287, -0.35762021931374804, -0.48943194575131316, -0.2934931849247994], [0.7188039915917864, 0.03169545156309653, -0.0714972263747659, -0.016048786123410482, -0.6813325146522308, 0.05166591672547598, 0.014268195229302044, -0.09667116076456264, 0.02268360453106016], [0.6926705741651878, -0.024611287396482218, 0.04005443367276633, 0.04504019151133196, 0.7111751309783907, -0.08191112156437473, 0.025763442295078168, 0.053147425820609995, 0.0009386808931254812]], 'd_pca': 4, 'n_tr': 960, 'ztr': '...'}, 'ds23': {'d_in': 10, 'norm': 'minmax', 'nprm': [[-0.9835869467132689, -1.1236929300157916, -1.0518588647998877, -0.4507205952337788, -0.4352936671415712, -0.9864501812277006, -0.8479559100263763, -0.8659134783455303, -0.9183708599801748, -0.5225513901602551], [0.4566207119080011, 0.574099770956434, 0.6415503154185513, 1.0812584238928398, 1.1802548111201514, 0.5354693214959577, 0.5905492391472807, 0.7753251927704521, 0.4256835861703724, 1.1312832875289172]], 'pmean': [2.168165253954151, 1.559371065526789, 1.2789619773427592, 0.4219340741810542, 0.373649910121186, 2.0421608489670904, 1.4988012228203573, 0.8109253842486372, 2.7384441992420165, 0.36998398002101096], 'pcomp': [[0.49837185780487925, 0.12881396656864352, -0.21688425060947647, -0.0035378593206142495, 0.003977425412610682, -0.3474589845965888, 0.36311304337018774, 0.1144301023421907, 0.6495218345995595, -0.02033835428374553], [0.5228935960089534, 0.5903381387374674, -0.15026634496552138, 0.004687373526240601, 0.001912845585577558, 0.42736166839033535, 0.059396130278297696, 0.11572723580713336, -0.3919210400245713, 0.04813287549890187], [0.16470585278448868, -0.5458095709312117, -0.4787977732245716, -0.008339044654424345, -0.0071740392899709085, 0.6077589044662574, -0.007739905816930228, -0.16850735507379602, 0.17711157131454622, -0.12806853899602272]], 'd_pca': 3, 'n_tr': 1008, 'ztr': '...'}}


def _uhex(s, shape):
    # hex of raw float32 bytes (float noise is incompressible, so no zlib/gzip
    # -- only numpy + builtins are needed to decode)
    return np.frombuffer(bytes.fromhex(s), dtype=np.float32).copy().reshape(shape)


def _clean(X):
    X = np.asarray(X, dtype=float)
    keep = ~np.all(np.isnan(X), axis=0)
    X = X[:, keep] if keep.any() else np.zeros((len(X), 1))
    for j in range(X.shape[1]):
        col = X[:, j]
        bad = np.isnan(col)
        if bad.any():
            X[bad, j] = np.nanmean(col)
    return X


def _fit_norm_params(X, method):
    """Parameters of the per-dataset normalization fitted on public train."""
    if method == "standard":
        s = StandardScaler().fit(X)
        return [s.mean_.tolist(), s.scale_.tolist()]
    if method == "robust":
        s = RobustScaler().fit(X)
        return [s.center_.tolist(), s.scale_.tolist()]
    if method == "minmax":
        s = MinMaxScaler().fit(X)
        return [s.data_min_.tolist(), s.scale_.tolist()]
    # log-sym: signed log compresses heavy tails, then standardize
    Xa = np.log1p(np.abs(X)) * np.sign(X)
    s = StandardScaler().fit(Xa)
    return [s.mean_.tolist(), s.scale_.tolist()]


def _apply_norm(X, method, prm):
    if method == "log-sym":
        X = np.log1p(np.abs(X)) * np.sign(X)
    mean, scale = np.asarray(prm[0]), np.asarray(prm[1])
    return (X - mean) / scale


def _fit_pca(Z):
    if Z.shape[1] <= 3:
        return Z.shape[1], np.zeros(Z.shape[1]), np.eye(Z.shape[1])
    p = PCA(random_state=42).fit(Z)
    cum = np.cumsum(p.explained_variance_ratio_)
    n = max(2, min(8, int(np.searchsorted(cum, 0.95) + 1), Z.shape[1]))
    return n, p.mean_[:].copy(), p.components_[:n].copy()


def _pca_transform(Z, d_pca, pmean, pcomp):
    if d_pca == Z.shape[1] and np.allclose(pcomp, np.eye(d_pca)):
        return Z - pmean
    return (Z - pmean) @ pcomp.T


def _sil(Z, lab):
    try:
        u = np.unique(lab)
        if len(u) < 2 or len(u) >= len(Z):
            return -1.0
        return float(silhouette_score(Z, lab))
    except Exception:
        return -1.0


def _consensus(cands):
    out = []
    for i, a in enumerate(cands):
        vals = [normalized_mutual_info_score(a, b)
                for j, b in enumerate(cands) if j != i]
        out.append(float(np.mean(vals)) if vals else 0.0)
    return np.array(out)


def _nn_dist(Z, k=5):
    return cKDTree(Z).query(Z, k=k + 1)[0][:, k]


def _core_mask(Z):
    """DBSCAN core-point idea: keep points whose kth-NN distance is not an outlier."""
    n = len(Z)
    k = 5 if n >= 200 else 3
    d = _nn_dist(Z, k)
    thr = max(np.quantile(d, 0.95), np.median(d) * 2.0)
    m = d <= thr
    frac = float(1.0 - m.mean())
    if not (0.003 <= frac <= 0.08):      # no clear tail -> do not trim
        return np.ones(n, dtype=bool), 0.0
    return m, frac


def _reassign_noise(Z, lab):
    out = lab.copy()
    ni = np.where(lab < 0)[0]
    if len(ni) and (lab >= 0).any():
        uniq = np.unique(lab[lab >= 0])
        cents = np.array([Z[lab == c].mean(0) for c in uniq])
        D = ((Z[ni][:, None, :] - cents[None, :, :]) ** 2).sum(-1)
        out[ni] = uniq[np.argmin(D, axis=1)]
    return out


def _extend(Zfit, lab_fit, Zall):
    """Label points outside the fit set by nearest cluster centroid in Z-space."""
    if len(Zall) == len(Zfit):
        return np.asarray(lab_fit, dtype=int)
    uniq = np.unique(lab_fit)
    cents = np.array([Zfit[lab_fit == c].mean(0) for c in uniq])
    D = ((Zall[:, None, :] - cents[None, :, :]) ** 2).sum(-1)
    return uniq[np.argmin(D, axis=1)]


def _pipeline(Z_tr, Z_te, ds):
    """Full ensemble on the pooled set. Returns (labels_pool, info)."""
    n_tr = len(Z_tr)
    Z = np.vstack([Z_tr, Z_te]) if n_tr else Z_te
    n = len(Z)
    k0 = K_PRIOR.get(ds, 3)
    ks = sorted({max(2, k0 - 1), k0, min(k0 + 1, max(2, n // 5))})

    # --- choose k: prior first; override only on a strong silhouette signal ---
    sils = {}
    for k in ks:
        sils[k] = _sil(Z, KMeans(n_clusters=k, n_init=15, random_state=42).fit(Z).labels_)
    best_k = max(ks, key=lambda k: -sils[k])
    if best_k != k0 and sils[best_k] - sils[k0] < 0.04:
        best_k = k0  # e.g. keep ring+center as 2, don't split the ring
    best_k = min(best_k, max(2, n // 3))

    # --- stability probe + density-based core set (on pooled data) ---
    fits = [KMeans(n_clusters=best_k, n_init=15, random_state=s).fit(Z)
            for s in (42, 7, 123)]
    stab = float(np.mean([adjusted_rand_score(fits[i].labels_, fits[j].labels_)
                          for i, j in ((0, 1), (0, 2), (1, 2))]))
    core_m, frac_rm = _core_mask(Z)
    use_core = stab < 0.8 and frac_rm > 0
    Zfit = Z[core_m] if use_core else Z

    # --- ensemble at best_k ---
    cands = []
    km = KMeans(n_clusters=best_k, n_init=15, random_state=42).fit(Zfit)
    cands.append(('KMeans', _extend(Zfit, km.labels_, Z)))
    try:
        gm = GaussianMixture(n_components=best_k, covariance_type="full",
                             n_init=3, random_state=42).fit(Zfit)
        lab_gm = gm.predict(Z)
        if len(np.unique(lab_gm)) == best_k:
            cands.append(('GaussianMixture', lab_gm))
    except Exception:
        pass
    try:
        sp = SpectralClustering(n_clusters=best_k, affinity="nearest_neighbors",
                                n_neighbors=min(len(Zfit) - 1,
                                                max(10, int(np.sqrt(len(Zfit))))),
                                random_state=42).fit_predict(Zfit)
        if len(np.unique(sp)) == best_k:
            cands.append(('SpectralClustering', _extend(Zfit, sp, Z)))
    except Exception:
        pass

    # --- DBSCAN with auto eps (confirming candidate only) ---
    d5 = _nn_dist(Z, 5 if n >= 200 else 3)
    db_lab = None
    for eps in np.unique(np.quantile(d5, [0.60, 0.75, 0.90, 0.95])):
        for ms in (max(3, n // 100), 5):
            lab = DBSCAN(eps=float(eps), min_samples=ms).fit_predict(Z)
            ncl = len(np.unique(lab)) - (1 if (lab < 0).any() else 0)
            noise = float((lab < 0).mean())
            if 2 <= ncl <= max(3, k0 + 3) and noise <= 0.15:
                lab_r = _reassign_noise(Z, lab)
                if db_lab is None or _sil(Z, lab_r) > _sil(Z, db_lab):
                    db_lab = lab_r
    if db_lab is not None and max(
            normalized_mutual_info_score(db_lab, lab) for _, lab in cands) >= 0.5:
        cands.append(('DBSCAN', db_lab))

    # --- HDBSCAN (confirming candidate only) ---
    try:
        hd = HDBSCAN(min_cluster_size=max(10, n // (4 * best_k))).fit_predict(Z)
        ncl = len(np.unique(hd)) - (1 if (hd < 0).any() else 0)
        noise = float((hd < 0).mean())
        if 2 <= ncl and noise <= 0.35:
            hd_r = _reassign_noise(Z, hd)
            if max(normalized_mutual_info_score(hd_r, lab) for _, lab in cands) >= 0.5:
                cands.append(('HDBSCAN', hd_r))
    except Exception:
        pass

    score = np.array([_sil(Z, c) for _, c in cands]) + 0.3 * _consensus([c for _, c in cands])
    i = int(np.argmax(score))
    labels = np.asarray(cands[i][1], dtype=int)
    info = {
        'k_prior': k0,
        'k_used': best_k,
        'sil_by_k': {k: round(float(s), 3) for k, s in sils.items()},
        'km_stability': round(stab, 3),
        'core_frac_removed': round(frac_rm, 3),
        'fit_set': 'core' if use_core else 'all',
        'pooled': int(n_tr),
        'methods': {name: round(float(s), 3) for (name, _), s in zip(cands, score)},
        'winner': cands[i][0],
    }

    # ds16/ds23: ring + central blob geometry beats arc-splitting
    if best_k == 2 and ds in RING_DS:
        rb = _ring_blob_labels(Z)
        if rb is not None:
            info['winner'] = 'ring/blob'
            info['methods'] = {**info['methods'],
                               'ring/blob': round(_sil(Z, rb), 3)}
            return rb, info
    return labels, info


def cluster(X, dataset_id=None):
    """Return labels only (grading API). Use explain() for the full report."""
    lab, _ = _run(X, dataset_id)
    return lab


def explain(X, dataset_id=None):
    """Same pipeline as cluster(); returns dict with 'labels' and diagnostics."""
    lab, info = _run(X, dataset_id)
    out = {'labels': lab}
    out.update(info)
    return out


def _run(X, dataset_id=None):
    Xc = _clean(X)
    n = len(Xc)
    ds = str(dataset_id or "").lower().strip("'\" ")
    art = ART.get(ds) if ART is not None else None
    if art is not None and Xc.shape[1] == art['d_in']:
        # transform test points into the train's Z-space with saved parameters
        Z0 = _apply_norm(Xc, art['norm'], art['nprm'])
        Z0 = np.where(np.isfinite(Z0), Z0, 0.0)
        Z_te = _pca_transform(Z0, art['d_pca'], np.asarray(art['pmean']), np.asarray(art['pcomp']))
        Z_tr = _uhex(art['ztr'], (art['n_tr'], art['d_pca']))
    else:
        # fallback: unknown dataset -> refit locally, cluster the given points only
        method = NORM_PRIOR.get(ds, "minmax")
        Z0 = _apply_norm(Xc, method, _fit_norm_params(Xc, method))
        Z0 = np.where(np.isfinite(Z0), Z0, 0.0)
        d_pca, pmean, pcomp = _fit_pca(Z0)
        Z_te = _pca_transform(Z0, d_pca, pmean, pcomp)
        Z_tr = np.zeros((0, d_pca))
    if n < 4 and len(Z_tr) < 4:
        lab = np.zeros(n, dtype=int)
        info = {'k_prior': None, 'k_used': 1, 'sil_by_k': {},
                'km_stability': 1.0, 'core_frac_removed': 0.0, 'fit_set': 'all',
                'pooled': int(len(Z_tr)), 'methods': {'constant': 0.0}, 'winner': 'constant'}
        return lab, info
    labels_pool, info = _pipeline(Z_tr, Z_te, ds)
    return np.asarray(labels_pool[len(Z_tr):], dtype=int), info


# --- ds16/ds23: ellipse ring + central blob fix (validated in sol_16_23/) ---
# Standard k=2 clusterers split these datasets into arcs; here we fit an
# ellipse-aware radial boundary in the top-2 Z dimensions and label
# inside/outside instead. Fitted on pooled data at call time, so it is more
# stable than fitting on ~250 test points alone.


def _ellipse_boundary(p):
    """Ellipse-aware ring boundary: per-angle median radius of the outer shell."""
    c0 = p.mean(0)
    d0 = np.linalg.norm(p - c0, axis=1)
    outer = d0 >= np.quantile(d0, 0.45)          # keep the outer shell only
    ang = np.arctan2(p[:, 1] - c0[1], p[:, 0] - c0[0])
    nb = 36
    edges = np.linspace(-np.pi, np.pi, nb + 1)
    Rb = np.full(nb, np.nan)
    for i, (e, f) in enumerate(zip(edges[:-1], edges[1:])):
        m = (ang >= e) & (ang < f) & outer
        if m.any():
            Rb[i] = np.median(d0[m])             # median = robust to blob leaks
    for i in range(nb):                          # fill any empty bins
        if np.isnan(Rb[i]):
            j = (i + 1) % nb
            while np.isnan(Rb[j]):
                j = (j + 1) % nb
            Rb[i] = Rb[j]
    k = np.array([1, 2, 3, 2, 1], float); k /= k.sum()
    sm = np.array([sum(k[j] * Rb[(i - j) % nb] for j in range(5)) for i in range(nb)])
    th = (edges[:-1] + edges[1:]) / 2
    return c0, th, sm


def _inside_boundary(p, c, th, R, frac=0.80):
    """True if point is clearly inside the fitted boundary."""
    d = np.linalg.norm(p - c, axis=1)
    a = (np.arctan2(p[:, 1] - c[1], p[:, 0] - c[0]) + np.pi) % (2 * np.pi) - np.pi
    t_ext = np.concatenate([th - 2 * np.pi, th, th + 2 * np.pi])
    R_ext = np.concatenate([R, R, R])
    Rloc = np.interp(a, t_ext, R_ext)
    return d < frac * Rloc


def _ring_blob_labels(Z):
    """Reference ring/blob split in Z[:, :2]; None if the fit degenerates."""
    p = Z[:, :2]
    c, th, R = _ellipse_boundary(p)
    lab = _inside_boundary(p, c, th, R).astype(int)
    frac = float(lab.mean())
    if not 0.15 < frac < 0.85:                  # degenerate fit -> fall back
        return None
    return lab


## Анализ на публичных данных

In [ ]:
# Анализ: прогон v4-конвейера по каждому датасету на публичных данных
import time
res = {}
for ds, g in df.groupby('dataset_id'):
    X = g[xcols].values.astype(float)
    t0 = time.time()
    lab, info = _run(X, ds)
    res[ds] = (info['k_used'], info['winner'], round(time.time() - t0, 2),
               int(len(set(lab))))
for ds in sorted(res):
    print(ds, 'k=%d' % res[ds][0], res[ds][1], 'rt=%.2fs' % res[ds][2],
          'labels=%d' % res[ds][3])

## Выводы

1. **Трансдуктивный подход.** Кластеризация на объединённом множестве «публичный трейн + тест»
   (~1200+ точек вместо ~244) делает выбор k и победителя ансамбля заметно стабильнее:
   в holdout-симуляции (80/20 сплит публичного трейна, артефакты сфитированы только на 80%)
   средний ARI между v3 и v4 = 0.72, а на большинстве датасетов k и winner совпадают
   во всех повторностях.
2. **Артефакты без меток.** Все встроённые параметры (нормализация, PCA, Z_tr) получены
   только из unlabeled `public_data/train.csv`; тестовые метки нигде не используются,
   скрытые данные не запрашиваются.
3. **Ограничения checker'а учтены.** Только разрешённые библиотеки (numpy/scipy/sklearn/pandas);
   стандартные модули не импортируются (артефакты закодированы hex-строками, декодирование —
   `bytes.fromhex` + numpy). Время работы: ~0.2–0.9 с на датасет.
4. **Слабые места.** ds13/ds14 — геометрия близка к шуму (низкий ARI v3/v4): решение опирается
   на a-priori k=2 и ансамблевый консенсус; ds08 — нестабильный спектральный кандидат, но окно k
   и consensus NMI сглаживают выбор.